# 🕵️ VELVET ENVELOPE
### An AI-Generated Detective Investigation

**Phase 1 — Architecture & Backend** · **Phase 2 — UI & Investigation Flow** · **Phase 3 — Puzzle Engine & Verdict System** · **Phase 4 — Polish**

The full game: dynamically loaded suspects/victims/puzzles from GitHub, an LLM-generated hidden case, a noir-styled Investigation Hub, a data-driven puzzle engine that gates evidence, and a Verdict Engine that scores the player's accusation and reveals the true story.

Run every cell in order once, top to bottom (Configuration will prompt for an OpenRouter API key). Then use the app itself — no re-running cells is needed, including to start a brand new investigation.

## 1 · Imports

All third-party dependencies used across the entire notebook live here so
later sections never need their own `import` statements.

In [1]:
# --- Standard library -----------------------------------------------------
import io
import json
import random
import re
import textwrap
import threading
import time
import getpass
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

# --- Third-party -----------------------------------------------------------
import requests
from IPython.display import Image, display, clear_output

# ipywidgets is used starting in Phase 2, imported here so the environment
# is validated early.
import ipywidgets as widgets

print("✅ Imports loaded.")


✅ Imports loaded.


## 2 · Configuration

Central place for every constant, credential, and tunable value.
Nothing below this cell should contain a hardcoded magic string that
belongs here instead.

In [2]:
# --- GitHub asset repository ------------------------------------------------
from google.colab import userdata
GITHUB_OWNER = "charishma-rai"
GITHUB_REPO = "uuuu"
GITHUB_BRANCH = "main"
ASSET_ROOT = "VelvetEnvelopeAssets/assets"       # suspects/ + victims/
PUZZLE_ROOT = "VelvetEnvelopeAssets/puzzles"      # cipher/, logic/, ...

GITHUB_API_BASE = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents"
GITHUB_RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_BRANCH}"

# --- OpenRouter (LLM) --------------------------------------------------------
# The key is requested interactively so it is never hardcoded or committed.
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
OPENROUTER_MODEL = "openai/gpt-4o-mini"   # swap for any OpenRouter model id
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_TEMPERATURE = 0.9
OPENROUTER_MAX_RETRIES = 2          # retries on invalid JSON before failing

# --- Pollinations (image generation) ----------------------------------------
POLLINATIONS_URL = "https://image.pollinations.ai/prompt/{prompt}"
CRIME_SCENE_FILENAME = "crime_scene.jpg"
CRIME_SCENE_SEED = None  # None -> random seed each run

# --- Gameplay rules -----------------------------------------------------------
NUM_SUSPECTS = 3
NUM_VICTIMS = 1
# NOTE: live suspect questioning (MAX_QUESTIONS_PER_SUSPECT / MAX_DIALOGUE_CALLS)
# has been removed. Every suspect's interrogation record is now generated
# once, up front, as part of the case itself -- see Section 8's
# "investigation_notes" and Section 11.
INVESTIGATION_NOTES_MIN = 4
INVESTIGATION_NOTES_MAX = 6

print("✅ Configuration set.")

✅ Configuration set.


## 3 · GitHub Loader

Everything that talks to the GitHub REST API lives here. Nothing else in
the notebook should call `requests.get` against GitHub directly — every
other section goes through these functions.

Design rules being followed:

* **Nothing is hardcoded.** Filenames are discovered by listing each
  directory through the GitHub Contents API.
* Suspects and victims are paired by shared numeric id
  (`suspect_001.jpeg` ↔ `suspect_001.json`).
* Puzzle categories are discovered by listing `puzzles/` itself, so a new
  category folder added later needs zero notebook changes.
* Every JSON puzzle file inside a category is loaded automatically, so
  adding `matching_006.json` makes it available with zero notebook
  changes, exactly as required.

In [3]:
def _request_with_retries(
    url: str,
    *,
    retries: int = 4,
    backoff_seconds: float = 2.0,
    timeout: int = 60,
    **kwargs,
):
    """GET a URL, retrying on timeouts/connection errors with backoff.

    GitHub's raw content host occasionally times out or drops a
    connection under load. A single flaky request shouldn't crash the
    whole notebook, so every network call in this notebook that hits
    GitHub goes through this helper instead of calling requests.get
    directly.
    """
    last_error = None
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=timeout, **kwargs)
            response.raise_for_status()
            return response
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as error:
            last_error = error
            if attempt < retries - 1:
                time.sleep(backoff_seconds * (attempt + 1))
    raise last_error


def _github_list_dir(path: str) -> list[dict]:
    """Return the raw GitHub Contents API listing for a directory.

    Each entry is a dict with at least: name, path, type ("file"/"dir"),
    download_url (for files).
    """
    url = f"{GITHUB_API_BASE}/{path}?ref={GITHUB_BRANCH}"
    response = _request_with_retries(url)
    return response.json()


def _github_get_json(download_url: str) -> dict:
    """Fetch and parse a single JSON file from its raw download URL."""
    response = _request_with_retries(download_url)
    return response.json()


def _character_id_from_filename(filename: str) -> str:
    """"suspect_003.json" -> "003" | "victim_002.jpeg" -> "002" """
    stem = Path(filename).stem
    return stem.split("_")[-1]


def load_characters(kind: str) -> list[dict]:
    """Load every suspect or victim as {id, image_url, metadata}.

    kind must be "suspects" or "victims". Images and metadata files are
    paired purely by their shared numeric id, discovered dynamically —
    no filenames are ever assumed to exist.
    """
    directory = f"{ASSET_ROOT}/{kind}"
    entries = _github_list_dir(directory)

    images_by_id: dict[str, str] = {}
    json_by_id: dict[str, dict] = {}

    for entry in entries:
        if entry["type"] != "file":
            continue
        char_id = _character_id_from_filename(entry["name"])
        if entry["name"].lower().endswith((".jpeg", ".jpg", ".png")):
            images_by_id[char_id] = entry["download_url"]
        elif entry["name"].lower().endswith(".json"):
            json_by_id[char_id] = _github_get_json(entry["download_url"])

    characters = []
    for char_id, metadata in json_by_id.items():
        if char_id not in images_by_id:
            continue  # skip incomplete pairs rather than crash the game
        characters.append({
            "id": char_id,
            "image_url": images_by_id[char_id],
            "metadata": metadata,
        })
    return characters


def load_puzzle_categories() -> list[str]:
    """Discover every puzzle category by listing puzzles/ itself."""
    entries = _github_list_dir(PUZZLE_ROOT)
    return sorted(entry["name"] for entry in entries if entry["type"] == "dir")


def load_all_puzzles() -> dict[str, list[dict]]:
    """Load every puzzle JSON in every category.

    Returns {category: [ {"source_file": name, **puzzle_json}, ... ]}.
    Adding a new .json file to any category folder on GitHub makes it
    available here automatically — nothing is hardcoded.
    """
    puzzles_by_category: dict[str, list[dict]] = {}
    for category in load_puzzle_categories():
        entries = _github_list_dir(f"{PUZZLE_ROOT}/{category}")
        puzzles = []
        for entry in entries:
            if entry["type"] == "file" and entry["name"].lower().endswith(".json"):
                puzzle_json = _github_get_json(entry["download_url"])
                puzzle_json["source_file"] = entry["name"]
                puzzles.append(puzzle_json)
        puzzles_by_category[category] = puzzles
    return puzzles_by_category


print("✅ GitHub loader ready.")

✅ GitHub loader ready.


## 4 · Utilities

Small, dependency-free helper functions shared across sections: character
selection and safe JSON parsing.

**Design note on "compatible" characters:** the spec restricts character
selection to using only `gender` and `age` from metadata. Since no
explicit compatibility rule was specified, the assumption implemented
here is that suspects should be plausible adults roughly within the
victim's generation (±25 years) so the LLM-generated relationships read
naturally. Gender is *not* used as a filter (there's no story reason two
suspects can't share a gender) but is threaded through to the case
generator prompt so the LLM can write age/gender-aware dialogue. This
rule is isolated in one function — adjust `_is_compatible` freely without
touching anything else.

In [4]:
# Keep track of recently used characters across investigations.
#
# FIX (Problem 9): the old version used a hard "used" set per kind that,
# once every character had been cast, cleared itself completely in one
# shot. With a small asset pool (a handful of portraits) that reset fires
# constantly, so the same faces kept reappearing case after case. This
# tracks recency order instead: characters that have never been used are
# always preferred, then whichever were used longest ago. There's no bulk
# reset — the pool never goes fully "unrestricted" the way a cleared set
# does, so repeats only happen once literally every character has been
# used at least as recently as everyone else.
_VICTIM_RECENCY: list[str] = []    # ids, oldest-used-first; never-used ids aren't listed yet
_SUSPECT_RECENCY: list[str] = []


def _get_gender(metadata: dict) -> str:
    """Return normalized gender from metadata."""
    gender = metadata.get("gender", "unknown")
    return str(gender).strip().lower()


def _pick_avoiding_recent(all_items: list[dict], recency: list[str], count: int) -> list[dict]:
    """Choose `count` items, preferring ones never used, then whichever
    were used longest ago. `recency` is mutated in place: oldest-use-first,
    with the newly-chosen ids moved to the end (most-recently-used).
    """
    all_ids = [item["id"] for item in all_items]
    # Drop ids for characters that no longer exist in the repo (e.g. an
    # asset was removed from GitHub since the last run).
    recency[:] = [char_id for char_id in recency if char_id in all_ids]

    never_used = [char_id for char_id in all_ids if char_id not in recency]
    random.shuffle(never_used)
    priority_order = never_used + recency  # never-used first, then least-recently-used

    chosen_ids = priority_order[:count]
    chosen = [item for item in all_items if item["id"] in chosen_ids]
    random.shuffle(chosen)

    for char_id in chosen_ids:
        if char_id in recency:
            recency.remove(char_id)
        recency.append(char_id)  # now the most-recently-used

    return chosen


def select_cast(all_victims: list[dict], all_suspects: list[dict]) -> tuple[dict, list[dict], dict]:
    """Choose 1 victim and NUM_SUSPECTS suspects, always favoring whichever
    characters have gone longest without appearing (or have never
    appeared at all), so the roster doesn't keep repeating.
    """
    if not all_victims or len(all_suspects) < NUM_SUSPECTS:
        raise ValueError("Not enough characters loaded from GitHub to cast a case.")

    victim = _pick_avoiding_recent(all_victims, _VICTIM_RECENCY, 1)[0]
    suspects = _pick_avoiding_recent(all_suspects, _SUSPECT_RECENCY, NUM_SUSPECTS)

    # Build a compact cast description for the LLM
    cast_info = {
        "victim": {
            "id": str(victim["id"]),
            "gender": _get_gender(victim["metadata"])
        },
        "suspects": [
            {
                "id": str(s["id"]),
                "gender": _get_gender(s["metadata"])
            }
            for s in suspects
        ]
    }

    return victim, suspects, cast_info


def safe_parse_json(raw_text: str) -> dict:
    """Strip common LLM formatting mistakes (markdown fences, stray
    whitespace, trailing commas) and parse strict JSON. Raises
    json.JSONDecodeError if the text still isn't valid JSON after cleanup.

    FIX (Problem 14): a single small repair pass (dropping a trailing
    comma before a closing ] or }) is attempted before giving up. This is
    the single most common LLM JSON mistake, and fixing it locally avoids
    burning a whole extra retry round-trip to the LLM for something this
    mechanical.
    """
    text = raw_text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        # Drop a leading language tag like "json\n"
        if text.lower().startswith("json"):
            text = text[4:]
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        repaired = re.sub(r",\s*([\]}])", r"\1", text)
        return json.loads(repaired)


print("✅ Utilities ready.")


✅ Utilities ready.


## 5 · OpenRouter Wrapper

The single choke point for every LLM call in the game. Case generation
and suspect dialogue both call `call_llm()` — nothing constructs an
OpenRouter request anywhere else, which keeps the "max 3 dialogue calls"
rule easy to enforce later in Section 11.

In [5]:
def call_llm(system_prompt: str, user_prompt: str, temperature: float = OPENROUTER_TEMPERATURE) -> str:
    """Send one chat completion request to OpenRouter and return the raw
    text of the model's reply (no parsing — callers decide how to use it).
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": OPENROUTER_MODEL,
        "temperature": temperature,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    }
    response = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=120)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]


def call_llm_json(system_prompt: str, user_prompt: str, temperature: float = OPENROUTER_TEMPERATURE) -> dict:
    """Call the LLM and parse strict JSON out of the reply, retrying with
    an increasingly forceful reminder if the model returns malformed JSON
    or markdown.
    """
    attempt_prompt = user_prompt
    last_error = None

    for attempt in range(OPENROUTER_MAX_RETRIES + 1):
        raw = call_llm(system_prompt, attempt_prompt, temperature)
        try:
            return safe_parse_json(raw)
        except json.JSONDecodeError as error:
            last_error = error
            attempt_prompt = (
                user_prompt
                + "\n\nYour previous reply was not valid JSON. "
                  "Return ONLY raw JSON. No markdown, no code fences, "
                  "no commentary, no trailing commas."
            )
    raise ValueError(f"LLM did not return valid JSON after retries: {last_error}")


print("✅ OpenRouter wrapper ready.")

✅ OpenRouter wrapper ready.


## 6 · Pollinations Wrapper

Generates the single crime scene image. Suspect and victim portraits are
never generated here — those come from GitHub assets only.

In [6]:
def make_image(description: str, file_name: str = CRIME_SCENE_FILENAME, seed: Optional[int] = None) -> Path:
    """Generate an image with Pollinations and save it to disk.

    Returns the local Path the image was saved to. Does NOT display it.

    FIX (Problem 3): this used to call display(Image(...)) itself, which
    meant the crime scene image popped up automatically the instant it
    finished generating -- before the player had opened the Crime Scene
    tab at all. Rendering is now solely the Crime Scene panel's job
    (Section 13c-ii), which already displays this same file from
    `crime_scene_path` only when that tab is actually opened. The image
    itself is still generated and saved exactly as before -- only the
    automatic display was removed.
    """
    link = POLLINATIONS_URL.format(prompt=requests.utils.quote(description))
    options = {"nologo": "true"}
    if seed is not None:
        options["seed"] = seed

    last_error = None
    for attempt in range(3):
        try:
            response = requests.get(link, params=options, timeout=180)
            response.raise_for_status()
            break
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as error:
            last_error = error
            if attempt < 2:
                time.sleep(2.0 * (attempt + 1))
    else:
        raise last_error

    path = Path(file_name)
    path.write_bytes(response.content)
    return path


print("✅ Pollinations wrapper ready.")

✅ Pollinations wrapper ready.


## 7 · Memory

Two isolated memory objects:

* **`HiddenMemory`** — the ground truth (killer, motive, solution,
  every suspect's internal truth). Never rendered to a widget. Section
  13 (Frontend) must never read from this object except through the
  Verdict Engine's comparison logic.
* **`InvestigationMemory`** — everything the player has actually done
  and seen: unlocked evidence, solved puzzles, questions asked, progress.
  This is the only memory the UI is allowed to display freely.

In [7]:
@dataclass
class HiddenMemory:
    """Ground-truth case data. Never exposed directly to the UI."""
    case_json: dict = field(default_factory=dict)

    @property
    def killer(self) -> str:
        return self.case_json.get("killer", "")

    @property
    def motive(self) -> str:
        return self.case_json.get("motive", "")

    @property
    def weapon(self) -> str:
        return self.case_json.get("weapon", "")

    @property
    def hidden_solution(self) -> str:
        return self.case_json.get("hidden_solution", "")

    @property
    def final_solution(self) -> dict:
        """The single, complete, pre-written reveal (murderer, motive,
        how_it_happened, evidence_supporting). Generated once alongside
        the rest of the case -- there is no follow-up LLM call that
        produces or narrates this at verdict time.
        """
        return self.case_json.get("final_solution", {})

    @property
    def suspects(self) -> list[dict]:
        return self.case_json.get("suspects", [])

    @property
    def evidence(self) -> list[dict]:
        return self.case_json.get("evidence", [])

    def suspect_truth(self, suspect_name: str) -> Optional[dict]:
        for suspect in self.suspects:
            if suspect.get("name") == suspect_name:
                return suspect.get("internal_truth")
        return None

    def investigation_notes_for(self, suspect_name: str) -> list[dict]:
        """The pre-generated interrogation transcript for one suspect:
        a list of {"question": ..., "answer": ...} pairs written once at
        case-generation time. Replaces the old live "ask one question"
        system entirely -- there is no LLM call here, just a lookup.
        """
        for suspect in self.suspects:
            if suspect.get("name") == suspect_name:
                return suspect.get("investigation_notes", [])
        return []


@dataclass
class InvestigationMemory:
    """Player-visible progress. Safe to read from the UI at any time."""
    unlocked_evidence: set = field(default_factory=set)
    solved_puzzles: set = field(default_factory=set)     # puzzle source_file names, so a puzzle is never reused

    # --- Phase 3: puzzle engine state ---------------------------------------
    # The puzzle assigned to each evidence item, chosen once and kept stable
    # across repeated attempts (so a wrong answer doesn't re-roll the puzzle).
    evidence_puzzles: dict = field(default_factory=dict)   # {evidence_title: puzzle_dict}
    puzzle_attempts: dict = field(default_factory=dict)    # {evidence_title: int}
    puzzle_points: dict = field(default_factory=dict)      # {evidence_title: int earned}, only set once solved

    # --- Verdict engine state -------------------------------------------------
    # None until the player submits a verdict; then holds the result so the
    # reveal is stable if the player revisits the Verdict panel.
    verdict: Optional[dict] = None

    # --- Phase 3 helpers -----------------------------------------------------
    def assign_puzzle(self, evidence_title: str, puzzle: dict) -> None:
        self.evidence_puzzles[evidence_title] = puzzle

    def get_assigned_puzzle(self, evidence_title: str) -> Optional[dict]:
        return self.evidence_puzzles.get(evidence_title)

    def record_puzzle_attempt(self, evidence_title: str) -> int:
        self.puzzle_attempts[evidence_title] = self.puzzle_attempts.get(evidence_title, 0) + 1
        return self.puzzle_attempts[evidence_title]

    def mark_puzzle_solved(self, source_file: str) -> None:
        self.solved_puzzles.add(source_file)

    def unlock_evidence(self, evidence_title: str) -> None:
        self.unlocked_evidence.add(evidence_title)

    def record_puzzle_points(self, evidence_title: str, points: int) -> None:
        self.puzzle_points[evidence_title] = points

    @property
    def total_puzzle_points(self) -> int:
        return sum(self.puzzle_points.values())


# Global game state singletons, (re)created each time a new investigation starts.
hidden_memory: Optional[HiddenMemory] = None
investigation_memory: Optional[InvestigationMemory] = None
loaded_puzzles: Optional[dict] = None       # category -> [puzzle, ...], filled in Section 3
crime_scene_path: Optional[Path] = None

print("✅ Memory system ready.")


✅ Memory system ready.


## 8 · Case Generator

Builds the prompt, calls the LLM exactly once (plus retries only on
malformed JSON), validates the shape of the response, and stores the
result in `HiddenMemory`.

The prompt asks the LLM to assign a `puzzle_category` to every evidence
item, constrained to the categories actually discovered on GitHub in
Section 3 — so the LLM can never reference a puzzle category that
doesn't exist in the repository.

In [8]:
# =============================================================================
# Section 8 · Case Generator — LOCKED-MURDERER PIPELINE
# =============================================================================
# The case is no longer written in one LLM call that decides guilt somewhere
# in the middle of a wall of text. It is built in five ordered steps, each
# one a separate LLM call (except Step 2, which is pure Python), where every
# step after Step 2 is handed the already-chosen murderer and is only asked
# to stay consistent with it.
#
#   Step 1 — generate_characters()           victim + 3 suspects, no guilt
#   Step 2 — select_murderer()               PURE PYTHON. Locks the murderer.
#   Step 3 — build_murder()                  motive/method/timeline/opportunity/
#                                             cover-up/planted evidence/misleading
#                                             clues, written FOR the locked murderer
#   Step 4 — generate_investigation_content() crime scene, evidence, puzzles,
#                                             investigation notes, witness info
#   Step 5 — generate_final_explanation()     reads solution.murderer, never
#                                             re-decides it
#
# `solution.murderer`, set once in Step 2, is the single source of truth.
# Nothing downstream is allowed to overwrite it — every later step either
# receives it as a fixed input or is validated against it, and the final
# consistency check (bottom of this section) refuses to return a case where
# any generated field disagrees with it.


# --- Shared schema constants -------------------------------------------------

REQUIRED_VICTIM_FIELDS = [
    "name",
    "occupation",
    "background",
    "public_reputation",
    "daily_routine",
    "recent_events",
    "timeline",
]

# Step 1 fields only — guilt-dependent fields (is_lying, internal_truth,
# investigation_notes) are deliberately NOT here, because at Step 1 no
# murderer has been chosen yet and nothing guilt-dependent may be written.
REQUIRED_CHARACTER_SUSPECT_FIELDS = [
    "asset_id",
    "name",
    "occupation",
    "relationship",
    "personality",
    "background",
    "known_reputation",
    "habit",
    "quirk",
    "usual_clothing",
    "known_skills",
    "recent_interaction",
    "possible_secret",
    "noticed_by_others",
    "possible_motive",
    "possible_alibi",
]

# Fields added in Step 4, once the murderer is locked and every suspect's
# relationship to that fact (lying or not) is known.
REQUIRED_GUILT_SUSPECT_FIELDS = [
    "claimed_location",
    "claimed_activity",
    "initial_statement",
    "is_lying",
    "internal_truth",
    "investigation_notes",
]

REQUIRED_EVIDENCE_FIELDS = [
    "title",
    "description",
    "relevance",
    "linked_suspect",
    "puzzle_category",
    "is_hidden_clue",
    "is_red_herring",
]

REQUIRED_CRIME_SCENE_FIELDS = [
    "location",
    "time_of_day",
    "weather",
    "lighting",
    "mood",
    "camera_angle",
    "victim_position",
    "environmental_storytelling",
    "objects",
    "important_clues",
    "hidden_clues",
    "red_herring_objects",
    "color_palette",
    "art_style",
    "image_prompt",
]

REQUIRED_BRIEFING_FIELDS = [
    "estimated_time_of_death",
    "how_body_was_discovered",
    "apparent_cause_of_death",
    "first_responding_officer",
]

REQUIRED_MURDER_FIELDS = [
    "motive",
    "weapon",
    "murder_method",
    "murder_timeline",
    "opportunity",
    "cover_up",
    "planted_evidence",
    "misleading_clues",
]

REQUIRED_FINAL_SOLUTION_FIELDS = [
    "murderer",
    "motive",
    "murder_method",
    "how_it_happened",
    "evidence_supporting",
    "why_others_innocent",
]

MIN_EVIDENCE_ITEMS = 5
MAX_EVIDENCE_ITEMS = 6


def _normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", (name or "").lower())


def _missing(obj: dict, required: list[str]) -> list[str]:
    return [f for f in required if f not in obj]


# =============================================================================
# STEP 1 — Generate Characters (victim + 3 suspects, guilt undecided)
# =============================================================================

CHARACTER_SYSTEM_PROMPT = """You are the character-writing engine for Velvet \
Envelope, a premium detective mystery game.

Your ONLY job right now is to invent a victim and three suspects who could \
plausibly appear in a murder mystery. You are NOT told who the murderer is \
because nobody has decided that yet — that happens in a separate step, \
after you finish. Do not write anything that assumes any particular \
suspect is guilty or innocent. Every suspect must be written as equally \
capable of having done it.

Every suspect needs a distinct personality, occupation, habit, and secret. \
Give each suspect a "possible_motive" (something that COULD explain a \
grudge or gain, whether or not it turns out to be the real one) and a \
"possible_alibi" (a claim about where they say they were), but do not \
decide which suspect's motive is real.

Return ONLY strict valid JSON. Never use markdown, code fences, or trailing \
commas. The JSON must be directly parseable by Python json.loads().
"""


def _build_character_prompt(
    victim: dict,
    suspects: list[dict],
    retry_feedback: Optional[str] = None,
) -> str:
    victim_meta = victim["metadata"]
    suspects_meta = [s["metadata"] for s in suspects]

    victim_fields = {
        "asset_id": str(victim["id"]),
        "gender": victim_meta.get("gender"),
        "age": victim_meta.get("age"),
    }
    suspect_fields = [
        {"asset_id": str(s["id"]), "gender": m.get("gender"), "age": m.get("age")}
        for s, m in zip(suspects, suspects_meta)
    ]

    retry_block = ""
    if retry_feedback:
        retry_block = textwrap.dedent(f"""
            Your previous attempt was REJECTED by an automated validator.
            Fix this exact problem and do not repeat it:

            {retry_feedback}

            Generate a corrected, complete set of characters from scratch.
        """).strip() + "\n\n"

    return retry_block + textwrap.dedent(f"""
        Generate ONE victim and exactly three suspects as JSON.

        Victim portrait metadata (asset_id/gender/age only, everything else
        is yours to invent): {json.dumps(victim_fields)}

        Suspect portraits metadata, in order, 3 suspects (asset_id/gender/age
        only): {json.dumps(suspect_fields)}

        Your "suspects" array MUST contain exactly these 3 asset_ids, one
        per suspect object, in any order — every asset_id supplied above
        MUST appear exactly once, copied verbatim onto "asset_id". Never
        invent, renumber, or drop an asset_id. Never change the supplied
        gender.

        The victim and every suspect must be distinct people with distinct
        names — never reuse the victim's name for a suspect.

        Return STRICT VALID JSON with EXACTLY this structure:

        {{
          "victim": {{
            "name": string,
            "occupation": string,
            "background": string,
            "public_reputation": string,
            "daily_routine": string,
            "recent_events": string,
            "timeline": string
          }},
          "suspects": [
            {{
              "asset_id": string,
              "name": string,
              "occupation": string,
              "relationship": string,
              "personality": string,
              "background": string,
              "known_reputation": string,
              "habit": string,
              "quirk": string,
              "usual_clothing": string,
              "known_skills": [string],
              "recent_interaction": string,
              "possible_secret": string,
              "noticed_by_others": string,
              "possible_motive": string,
              "possible_alibi": string
            }}
            // exactly 3 suspects
          ]
        }}

        RULES

        • No repeated occupations, habits, or personalities across suspects.
        • Every suspect should have a secret unrelated to the murder.
        • Recent interactions with the victim should naturally hint at
          possible motives without confirming any of them.
        • Do not decide or imply who the murderer is anywhere in this
          output.
    """).strip()


def _validate_characters_json(
    characters: dict,
    expected_asset_ids: list[str],
) -> None:
    for key in ("victim", "suspects"):
        if key not in characters:
            raise ValueError(f"Characters JSON missing top-level key: '{key}'. Add it.")

    victim_missing = _missing(characters["victim"], REQUIRED_VICTIM_FIELDS)
    if victim_missing:
        raise ValueError(f"Victim object is missing required field(s): {victim_missing}.")

    if len(characters["suspects"]) != NUM_SUSPECTS:
        raise ValueError(
            f"Characters JSON must contain exactly {NUM_SUSPECTS} suspects, "
            f"got {len(characters['suspects'])}."
        )

    returned_asset_ids = [str(s.get("asset_id", "")) for s in characters["suspects"]]
    if sorted(returned_asset_ids) != sorted(expected_asset_ids):
        raise ValueError(
            f"Suspect asset_ids must be exactly {expected_asset_ids} (each "
            f"used once, any order), got {returned_asset_ids}. Copy the "
            f"asset_id from the supplied portrait metadata onto each "
            f"suspect — never invent one."
        )

    for suspect in characters["suspects"]:
        missing = _missing(suspect, REQUIRED_CHARACTER_SUSPECT_FIELDS)
        if missing:
            raise ValueError(
                f"Suspect '{suspect.get('name', '?')}' is missing required "
                f"field(s): {missing}."
            )

    victim_name_normalized = _normalize_name(characters["victim"].get("name", ""))
    for suspect in characters["suspects"]:
        if victim_name_normalized and _normalize_name(suspect.get("name", "")) == victim_name_normalized:
            raise ValueError(
                f"Suspect '{suspect.get('name', '')}' has the exact same "
                f"name as the victim. Rename this suspect."
            )

    names_normalized = [_normalize_name(s.get("name", "")) for s in characters["suspects"]]
    if len(set(names_normalized)) != len(names_normalized):
        raise ValueError("Two suspects were given the same name. Every suspect needs a distinct name.")


def generate_characters(victim: dict, suspects: list[dict]) -> dict:
    """STEP 1 — generate the victim and three suspects. Guilt is not decided
    here and must not be implied anywhere in the output."""
    expected_asset_ids = [str(s["id"]) for s in suspects]

    characters = None
    last_error = None
    retry_feedback = None

    for _attempt in range(OPENROUTER_MAX_RETRIES + 1):
        user_prompt = _build_character_prompt(victim, suspects, retry_feedback=retry_feedback)
        candidate = call_llm_json(CHARACTER_SYSTEM_PROMPT, user_prompt)
        try:
            _validate_characters_json(candidate, expected_asset_ids)
        except ValueError as error:
            last_error = error
            retry_feedback = str(error)
            continue
        characters = candidate
        break

    if characters is None:
        raise ValueError(f"Character JSON failed validation after retries: {last_error}")

    return characters


# =============================================================================
# STEP 2 — Select the Murderer (pure Python, no LLM call)
# =============================================================================

def select_murderer(characters: dict) -> dict:
    """STEP 2 — choose exactly one of the three already-generated suspects
    as the murderer, BEFORE any murder-specific content exists.

    This is deliberately not an LLM decision: picking in plain Python means
    the choice can never drift, be re-argued, or be second-guessed by a
    later generation step. The result is written into
    `characters["solution"]` immediately and is treated as fixed from this
    point forward — every later step receives it as an input, never as
    something to decide.
    """
    murderer = random.choice(characters["suspects"])
    solution = {
        "murderer": murderer["name"],
        "murderer_asset_id": str(murderer["asset_id"]),
    }
    characters["solution"] = solution
    return solution


# =============================================================================
# STEP 3 — Build the Murder (using the locked murderer)
# =============================================================================

MURDER_SYSTEM_PROMPT = """You are the murder-writing engine for Velvet \
Envelope, a premium detective mystery game.

You are given a cast of characters AND the identity of the murderer, which \
has already been decided by the game engine before you were called. This \
is fixed. You must build the murder around that exact person — never \
substitute, hint at, or favor a different suspect as the true killer.

Every other suspect must still look plausibly guilty on the surface: give \
them suspicious behaviour, secrets, and shaky-looking alibis that are \
ultimately unrelated to the real murder. The mystery must be solvable — \
fair, internally consistent, no contradictions — but not obvious.

Return ONLY strict valid JSON. Never use markdown, code fences, or trailing \
commas. The JSON must be directly parseable by Python json.loads().
"""


def _build_murder_prompt(
    characters: dict,
    solution: dict,
    retry_feedback: Optional[str] = None,
) -> str:
    retry_block = ""
    if retry_feedback:
        retry_block = textwrap.dedent(f"""
            Your previous attempt was REJECTED by an automated validator.
            Fix this exact problem and do not repeat it:

            {retry_feedback}

            Generate a corrected, complete answer from scratch.
        """).strip() + "\n\n"

    return retry_block + textwrap.dedent(f"""
        THE MURDERER IS ALREADY DECIDED. This is fixed and must not change:

        murderer name: {solution['murderer']}
        murderer asset_id: {solution['murderer_asset_id']}

        Victim: {json.dumps(characters['victim'])}

        Suspects (the murderer is one of these three — build the case
        around whichever one matches the locked name above): {json.dumps(characters['suspects'])}

        Using this locked murderer, write the mechanics of the murder as
        JSON with EXACTLY this structure:

        {{
          "case_title": string,
          "summary": string,
          "location": string,
          "motive": string,          // the real reason {solution['murderer']} did it
          "weapon": string,
          "murder_method": string,   // how the murder was physically carried out
          "murder_timeline": string, // chronological account of the murder itself
          "opportunity": string,     // why/how the murderer was alone with the victim
          "cover_up": string,        // what the murderer did afterward to hide it
          "planted_evidence": [string],  // 1-3 items the murderer planted or staged
          "misleading_clues": [string]   // 1-3 things that could wrongly implicate an innocent suspect
        }}

        RULES

        • Every field must be consistent with {solution['murderer']} being
          the murderer. Do not write anything that would make a different
          suspect the true killer.
        • "misleading_clues" must point at one or both of the OTHER two
          suspects, not at {solution['murderer']}.
        • Keep it believable and specific, not generic.
    """).strip()


def _validate_murder_json(murder: dict, solution: dict) -> None:
    missing = _missing(murder, REQUIRED_MURDER_FIELDS)
    if missing:
        raise ValueError(f"Murder JSON is missing required field(s): {missing}.")
    for key in ("planted_evidence", "misleading_clues"):
        if not isinstance(murder[key], list) or not murder[key]:
            raise ValueError(f"'{key}' must be a non-empty list of short strings.")
    for key in ("case_title", "summary", "location", "motive", "weapon", "murder_method", "murder_timeline", "opportunity", "cover_up"):
        if not str(murder.get(key, "")).strip():
            raise ValueError(f"'{key}' must not be empty.")


def build_murder(characters: dict, solution: dict) -> dict:
    """STEP 3 — using the murderer locked in Step 2, generate motive,
    method, timeline, opportunity, cover-up, planted evidence, and
    misleading clues. Everything here must be consistent with
    `solution['murderer']`."""
    murder = None
    last_error = None
    retry_feedback = None

    for _attempt in range(OPENROUTER_MAX_RETRIES + 1):
        user_prompt = _build_murder_prompt(characters, solution, retry_feedback=retry_feedback)
        candidate = call_llm_json(MURDER_SYSTEM_PROMPT, user_prompt)
        try:
            _validate_murder_json(candidate, solution)
        except ValueError as error:
            last_error = error
            retry_feedback = str(error)
            continue
        murder = candidate
        break

    if murder is None:
        raise ValueError(f"Murder JSON failed validation after retries: {last_error}")

    return murder


# =============================================================================
# STEP 4 — Generate Investigation Content (using the locked murderer)
# =============================================================================

INVESTIGATION_SYSTEM_PROMPT = """You are the investigation-content engine \
for Velvet Envelope, a premium detective mystery game.

You are given the cast, the locked murderer, and the finished murder \
mechanics (motive, method, timeline, opportunity, cover-up, planted \
evidence, misleading clues) — all already decided before you were called. \
Your job is to build everything the PLAYER sees while investigating: the \
crime scene, the evidence, the pre-written interrogation transcripts, and \
the intake briefing. Every clue you generate must ultimately support the \
same locked murderer — no contradictions, and no clue may accidentally \
prove a different suspect did it.

Return ONLY strict valid JSON. Never use markdown, code fences, or trailing \
commas. The JSON must be directly parseable by Python json.loads().
"""


def _build_investigation_prompt(
    characters: dict,
    solution: dict,
    murder: dict,
    puzzle_categories: list[str],
    retry_feedback: Optional[str] = None,
) -> str:
    retry_block = ""
    if retry_feedback:
        retry_block = textwrap.dedent(f"""
            Your previous attempt was REJECTED by an automated validator.
            Fix this exact problem and do not repeat it:

            {retry_feedback}

            Generate a corrected, complete answer from scratch.
        """).strip() + "\n\n"

    return retry_block + textwrap.dedent(f"""
        THE MURDERER IS ALREADY DECIDED. This is fixed and must not change:

        murderer name: {solution['murderer']}

        Victim: {json.dumps(characters['victim'])}
        Suspects: {json.dumps(characters['suspects'])}
        Murder mechanics (already decided, build everything below to agree
        with this exactly): {json.dumps(murder)}

        Available puzzle categories you MUST use when assigning puzzles to
        evidence (copy exactly, lowercase): {json.dumps(puzzle_categories)}

        Return STRICT VALID JSON with EXACTLY this structure:

        {{
          "suspects": [
            {{
              "asset_id": string,          // copy exactly from the supplied suspects
              "claimed_location": string,
              "claimed_activity": string,
              "initial_statement": string,
              "is_lying": boolean,         // true only for suspects hiding something about the murder
              "internal_truth": string,    // what actually happened for this suspect, ground truth
              "investigation_notes": [
                {{ "question": string, "answer": string }}
                // 4-6 question/answer pairs
              ]
            }}
            // one entry per suspect, matched by asset_id
          ],

          "evidence": [
            {{
              "title": string,
              "description": string,
              "relevance": string,
              "linked_suspect": string,   // exact victim/suspect name, or "none"
              "puzzle_category": string,  // exact lowercase category from the list above
              "is_hidden_clue": boolean,
              "is_red_herring": boolean
            }}
            // exactly 5-6 evidence items
          ],

          "crime_scene": {{
            "location": string,
            "time_of_day": string,
            "weather": string,
            "lighting": string,
            "mood": string,
            "camera_angle": string,
            "victim_position": string,
            "environmental_storytelling": string,
            "objects": [string],
            "important_clues": [string],
            "hidden_clues": [string],
            "red_herring_objects": [string],
            "color_palette": string,
            "art_style": string,
            "image_prompt": string
          }},

          "briefing": {{
            "estimated_time_of_death": string,
            "how_body_was_discovered": string,
            "apparent_cause_of_death": string,
            "first_responding_officer": string
          }}
        }}

        ONLY {solution['murderer']} may be genuinely guilty. Every other
        suspect's "is_lying" should reflect an innocent secret or evasion,
        never a confession-adjacent detail about the real murder.

        Investigation notes must read like real interview notes — plain,
        specific, evasive where the suspect is lying — and must NEVER
        directly name the killer or state the solution outright.

        Each evidence item's "linked_suspect" must exactly match one of the
        suspects' or the victim's name, or the literal word "none".

        The important planted/misleading detail from the murder mechanics
        above should surface again here, inside "evidence" and/or
        "crime_scene", so the case is actually solvable from what the
        player can see.

        Crime Scene Image Instructions

        "image_prompt" is REQUIRED and MUST NOT be empty. Write a highly
        detailed cinematic prompt in a realistic Victorian detective
        aesthetic with subtle noir atmosphere, describing only what the
        investigator would see upon entering the room (architecture,
        furniture, lighting, weather, textures, objects, evidence
        placement, composition, mood, camera angle, color palette). Include
        2-3 visible-but-unlabeled visual clues, at least one of which must
        also appear in "evidence". Include 1-2 convincing red herring
        objects. Do NOT name the murderer, narrate the murder, include
        names, or reveal the solution.

        The briefing must read like a case intake sheet: it MUST NOT name
        the killer, state the motive, or reveal any hidden clue.
    """).strip()


def _validate_investigation_json(
    investigation: dict,
    characters: dict,
    solution: dict,
    murder: dict,
    puzzle_categories: list[str],
) -> None:
    for key in ("suspects", "evidence", "crime_scene", "briefing"):
        if key not in investigation:
            raise ValueError(f"Investigation JSON missing top-level key: '{key}'. Add it.")

    expected_asset_ids = {str(s["asset_id"]) for s in characters["suspects"]}
    returned_asset_ids = {str(s.get("asset_id", "")) for s in investigation["suspects"]}
    if returned_asset_ids != expected_asset_ids:
        raise ValueError(
            f"'suspects' entries must have asset_id exactly matching the "
            f"cast {sorted(expected_asset_ids)}, got {sorted(returned_asset_ids)}."
        )

    guilty_count = 0
    for suspect in investigation["suspects"]:
        missing = _missing(suspect, REQUIRED_GUILT_SUSPECT_FIELDS)
        if missing:
            raise ValueError(
                f"Suspect asset_id '{suspect.get('asset_id', '?')}' is "
                f"missing required field(s): {missing}."
            )
        notes = suspect.get("investigation_notes")
        if not isinstance(notes, list) or not (INVESTIGATION_NOTES_MIN <= len(notes) <= INVESTIGATION_NOTES_MAX):
            got = len(notes) if isinstance(notes, list) else type(notes).__name__
            raise ValueError(
                f"Suspect asset_id '{suspect.get('asset_id', '?')}' "
                f"investigation_notes must be a list of "
                f"{INVESTIGATION_NOTES_MIN}-{INVESTIGATION_NOTES_MAX} "
                f"question/answer objects, got {got}."
            )
        for note in notes:
            question_ok = isinstance(note, dict) and bool(str(note.get("question", "")).strip())
            answer_ok = isinstance(note, dict) and bool(str(note.get("answer", "")).strip())
            if not question_ok or not answer_ok:
                raise ValueError(
                    f"Suspect asset_id '{suspect.get('asset_id', '?')}' has "
                    f"an investigation_notes entry missing a non-empty "
                    f"'question' or 'answer' field: {note!r}."
                )
        if str(suspect.get("asset_id", "")) == solution["murderer_asset_id"]:
            if not suspect.get("is_lying"):
                raise ValueError(
                    f"The locked murderer ({solution['murderer']}) must "
                    f"have is_lying=true — they are hiding the murder."
                )
            guilty_count += 1

    if guilty_count != 1:
        raise ValueError(
            f"Exactly one suspect (the locked murderer, "
            f"{solution['murderer']}) may be genuinely guilty; found "
            f"{guilty_count} matching the murderer's asset_id."
        )

    if not (MIN_EVIDENCE_ITEMS <= len(investigation["evidence"]) <= MAX_EVIDENCE_ITEMS):
        raise ValueError(
            f"Investigation JSON must contain {MIN_EVIDENCE_ITEMS}-"
            f"{MAX_EVIDENCE_ITEMS} evidence items, got {len(investigation['evidence'])}."
        )

    normalized_categories = {c.strip().lower(): c for c in puzzle_categories}
    valid_linked_names = {s["name"] for s in characters["suspects"]}
    valid_linked_names.add(characters["victim"].get("name", ""))
    valid_linked_normalized = {_normalize_name(n) for n in valid_linked_names}
    valid_linked_normalized.update({"none", "unknown"})

    for evidence in investigation["evidence"]:
        missing = _missing(evidence, REQUIRED_EVIDENCE_FIELDS)
        if missing:
            raise ValueError(f"Evidence '{evidence.get('title', '?')}' is missing required field(s): {missing}.")
        raw_category = evidence.get("puzzle_category", "")
        category = str(raw_category).strip().lower()
        if category not in normalized_categories:
            raise ValueError(
                f"Evidence '{evidence.get('title')}' uses unknown puzzle "
                f"category '{raw_category}'. Must be one of "
                f"{puzzle_categories} (case-insensitive)."
            )
        evidence["puzzle_category"] = normalized_categories[category]

        linked = evidence.get("linked_suspect", "")
        if _normalize_name(linked) not in valid_linked_normalized:
            raise ValueError(
                f"Evidence '{evidence.get('title')}' has linked_suspect "
                f"'{linked}', which doesn't match the victim's name, any "
                f"suspect's exact name, or 'none'."
            )

    crime_scene_missing = _missing(investigation["crime_scene"], REQUIRED_CRIME_SCENE_FIELDS)
    if crime_scene_missing:
        raise ValueError(f"crime_scene object is missing required field(s): {crime_scene_missing}.")
    if not investigation["crime_scene"].get("image_prompt", "").strip():
        raise ValueError("crime_scene.image_prompt is missing or empty.")

    briefing_missing = _missing(investigation["briefing"], REQUIRED_BRIEFING_FIELDS)
    if briefing_missing:
        raise ValueError(f"briefing object is missing required field(s): {briefing_missing}.")


def generate_investigation_content(
    characters: dict,
    solution: dict,
    murder: dict,
    puzzle_categories: list[str],
) -> dict:
    """STEP 4 — using the locked murderer and the finished murder mechanics,
    generate the crime scene, evidence, interrogation transcripts, and
    briefing. Every clue must ultimately support `solution['murderer']`."""
    investigation = None
    last_error = None
    retry_feedback = None

    for _attempt in range(OPENROUTER_MAX_RETRIES + 1):
        user_prompt = _build_investigation_prompt(
            characters, solution, murder, puzzle_categories, retry_feedback=retry_feedback
        )
        candidate = call_llm_json(INVESTIGATION_SYSTEM_PROMPT, user_prompt)
        try:
            _validate_investigation_json(candidate, characters, solution, murder, puzzle_categories)
        except ValueError as error:
            last_error = error
            retry_feedback = str(error)
            continue
        investigation = candidate
        break

    if investigation is None:
        raise ValueError(f"Investigation JSON failed validation after retries: {last_error}")

    return investigation


# =============================================================================
# STEP 5 — Generate the Final Detective Explanation
# =============================================================================

EXPLANATION_SYSTEM_PROMPT = """You are the reveal-writing engine for \
Velvet Envelope, a premium detective mystery game.

You do NOT decide who the murderer is — that was already decided before \
you were called, and it is given to you below. Your only job is to write \
the final explanation the detective delivers once the case is solved, \
naming that exact same murderer and no one else.

Return ONLY strict valid JSON. Never use markdown, code fences, or trailing \
commas. The JSON must be directly parseable by Python json.loads().
"""


def _build_explanation_prompt(
    characters: dict,
    solution: dict,
    murder: dict,
    investigation: dict,
    retry_feedback: Optional[str] = None,
) -> str:
    retry_block = ""
    if retry_feedback:
        retry_block = textwrap.dedent(f"""
            Your previous attempt was REJECTED by an automated validator.
            Fix this exact problem and do not repeat it:

            {retry_feedback}

            Generate a corrected, complete answer from scratch.
        """).strip() + "\n\n"

    other_suspects = [s["name"] for s in characters["suspects"] if s["name"] != solution["murderer"]]

    return retry_block + textwrap.dedent(f"""
        Read this value and do not change it: solution.murderer =
        "{solution['murderer']}". The explanation you write below MUST name
        exactly this person and no one else as the murderer.

        Murder mechanics (already decided): {json.dumps(murder)}
        Evidence available to the player: {json.dumps(investigation['evidence'])}
        Other suspects, who must be shown to be innocent: {json.dumps(other_suspects)}

        Return STRICT VALID JSON with EXACTLY this structure:

        {{
          "final_solution": {{
            "murderer": "{solution['murderer']}",
            "motive": string,
            "murder_method": string,
            "how_it_happened": string,       // 3-5 sentence chronological narrative
            "evidence_supporting": [string], // 3-5 bullet strings, each naming a real evidence title above and how it points to the murderer
            "why_others_innocent": [string]  // one short string per other suspect, explaining why they are not guilty
          }}
        }}

        This is the only explanation text the player will ever see, so it
        must be complete and satisfying to read on its own.
    """).strip()


def _validate_explanation_json(explanation: dict, characters: dict, solution: dict) -> None:
    if "final_solution" not in explanation or not isinstance(explanation["final_solution"], dict):
        raise ValueError("Explanation JSON must contain a 'final_solution' object.")

    final_solution = explanation["final_solution"]
    missing = _missing(final_solution, REQUIRED_FINAL_SOLUTION_FIELDS)
    if missing:
        raise ValueError(f"final_solution is missing required field(s): {missing}.")

    if _normalize_name(final_solution.get("murderer", "")) != _normalize_name(solution["murderer"]):
        raise ValueError(
            f"final_solution.murderer (\"{final_solution.get('murderer', '')}\") "
            f"must exactly match the locked murderer "
            f"(\"{solution['murderer']}\"). Never name a different person."
        )

    if not isinstance(final_solution.get("evidence_supporting"), list) or not final_solution["evidence_supporting"]:
        raise ValueError("final_solution.evidence_supporting must be a non-empty list of short strings.")

    other_suspects = [s["name"] for s in characters["suspects"] if s["name"] != solution["murderer"]]
    why_innocent = final_solution.get("why_others_innocent")
    if not isinstance(why_innocent, list) or len(why_innocent) < len(other_suspects):
        raise ValueError(
            f"final_solution.why_others_innocent must contain one entry "
            f"per non-murderer suspect ({other_suspects})."
        )


def generate_final_explanation(
    characters: dict,
    solution: dict,
    murder: dict,
    investigation: dict,
) -> dict:
    """STEP 5 — read the locked `solution['murderer']` and generate the
    complete detective explanation using that same murderer. Never
    generates or considers a different murderer."""
    explanation = None
    last_error = None
    retry_feedback = None

    for _attempt in range(OPENROUTER_MAX_RETRIES + 1):
        user_prompt = _build_explanation_prompt(
            characters, solution, murder, investigation, retry_feedback=retry_feedback
        )
        candidate = call_llm_json(EXPLANATION_SYSTEM_PROMPT, user_prompt)
        try:
            _validate_explanation_json(candidate, characters, solution)
        except ValueError as error:
            last_error = error
            retry_feedback = str(error)
            continue
        explanation = candidate
        break

    if explanation is None:
        raise ValueError(f"Explanation JSON failed validation after retries: {last_error}")

    # Force-sync rather than burn a retry on a near-miss name (whitespace,
    # casing) — the two fields describing the same person is a guarantee
    # the engine can just make true.
    explanation["final_solution"]["murderer"] = solution["murderer"]
    return explanation


# =============================================================================
# Assembly + Final Validation
# =============================================================================

def _assemble_case_json(
    characters: dict,
    solution: dict,
    murder: dict,
    investigation: dict,
    explanation: dict,
) -> dict:
    """Merge the five steps' output into the single case_json shape the
    rest of the app (HiddenMemory, all render_*_panel functions) expects.
    No new decisions are made here — this is pure reassembly."""
    suspects_by_id = {str(s["asset_id"]): dict(s) for s in characters["suspects"]}
    for guilt_entry in investigation["suspects"]:
        asset_id = str(guilt_entry["asset_id"])
        suspects_by_id[asset_id].update(guilt_entry)

    case_json = {
        "case_title": murder["case_title"],
        "summary": murder["summary"],
        "location": murder["location"],
        "timeline": murder["murder_timeline"],
        "weapon": murder["weapon"],
        "motive": murder["motive"],
        "hidden_solution": murder["cover_up"],

        # Back-compat top-level keys the rest of the app already reads.
        "killer": solution["murderer"],
        "killer_asset_id": solution["murderer_asset_id"],
        "solution": dict(solution),  # {"murderer": ..., "murderer_asset_id": ...}

        "murder_details": {
            "murder_method": murder["murder_method"],
            "opportunity": murder["opportunity"],
            "cover_up": murder["cover_up"],
            "planted_evidence": murder["planted_evidence"],
            "misleading_clues": murder["misleading_clues"],
        },

        "victim": dict(characters["victim"]),
        "suspects": list(suspects_by_id.values()),
        "evidence": investigation["evidence"],
        "crime_scene": investigation["crime_scene"],
        "briefing": investigation["briefing"],
        "final_solution": explanation["final_solution"],
    }
    return case_json


def _final_consistency_check(
    case_json: dict,
    puzzle_categories: list[str],
    expected_asset_ids: list[str],
) -> None:
    """Runs once, after all five steps, before the case is handed to
    HiddenMemory. This is the last gate — it re-verifies every guarantee
    the pipeline was supposed to hold, rather than trusting each step's
    own validation blindly."""

    # 1. The locked murderer never drifted.
    locked_murderer = case_json["solution"]["murderer"]
    if _normalize_name(case_json["killer"]) != _normalize_name(locked_murderer):
        raise ValueError("case_json.killer does not match solution.murderer.")
    if _normalize_name(case_json["final_solution"]["murderer"]) != _normalize_name(locked_murderer):
        raise ValueError("final_solution.murderer does not match solution.murderer.")

    suspect_names = {s["name"] for s in case_json["suspects"]}
    if locked_murderer not in suspect_names:
        raise ValueError("solution.murderer is not one of the generated suspects.")

    # 2. Exactly one suspect is guilty.
    guilty = [s for s in case_json["suspects"] if s.get("is_lying") and s["name"] == locked_murderer]
    if len(guilty) != 1:
        raise ValueError("Exactly one suspect must be the locked murderer with is_lying=true.")

    # 3. Every suspect has the full schema (character fields + guilt fields).
    for suspect in case_json["suspects"]:
        missing = _missing(suspect, REQUIRED_CHARACTER_SUSPECT_FIELDS + REQUIRED_GUILT_SUSPECT_FIELDS)
        if missing:
            raise ValueError(f"Suspect '{suspect.get('name', '?')}' is missing field(s) after assembly: {missing}.")

    # 4. Evidence, crime scene, briefing, victim all present and complete.
    victim_missing = _missing(case_json["victim"], REQUIRED_VICTIM_FIELDS)
    if victim_missing:
        raise ValueError(f"Victim is missing field(s) after assembly: {victim_missing}.")

    if not (MIN_EVIDENCE_ITEMS <= len(case_json["evidence"]) <= MAX_EVIDENCE_ITEMS):
        raise ValueError("Evidence count out of range after assembly.")

    normalized_categories = {c.strip().lower() for c in puzzle_categories}
    for evidence in case_json["evidence"]:
        if evidence.get("puzzle_category", "").strip().lower() not in normalized_categories:
            raise ValueError(f"Evidence '{evidence.get('title')}' has an unresolved puzzle_category after assembly.")

    if not case_json["crime_scene"].get("image_prompt", "").strip():
        raise ValueError("crime_scene.image_prompt is empty after assembly.")

    briefing_missing = _missing(case_json["briefing"], REQUIRED_BRIEFING_FIELDS)
    if briefing_missing:
        raise ValueError(f"briefing is missing field(s) after assembly: {briefing_missing}.")

    # 5. Investigation notes never name the killer outright.
    murderer_normalized = _normalize_name(locked_murderer)
    for suspect in case_json["suspects"]:
        for note in suspect.get("investigation_notes", []):
            answer_normalized = _normalize_name(note.get("answer", ""))
            if murderer_normalized and murderer_normalized in answer_normalized and suspect["name"] != locked_murderer:
                # An innocent suspect's answer literally contains the killer's
                # name as a run-on substring of the normalized text — most
                # likely an accidental confession-by-mention. Flag it so a
                # retry can fix the wording rather than silently shipping it.
                raise ValueError(
                    f"Suspect '{suspect['name']}' investigation_notes answer "
                    f"appears to directly name the murderer "
                    f"('{note.get('answer', '')}'). Rewrite it so it doesn't "
                    f"give away the solution."
                )

    # 6. Asset ids are exactly the ones supplied — nothing invented/dropped.
    returned_asset_ids = sorted(str(s.get("asset_id", "")) for s in case_json["suspects"])
    if returned_asset_ids != sorted(expected_asset_ids):
        raise ValueError("Suspect asset_ids do not match the supplied portraits after assembly.")


def generate_case(victim: dict, suspects: list[dict], puzzle_categories: list[str]) -> HiddenMemory:
    """Run the full five-step locked-murderer pipeline and return the
    finished, validated case as a HiddenMemory.

    STEP 1 -> STEP 2 (locks the murderer) -> STEP 3 -> STEP 4 -> STEP 5 ->
    final consistency check. The murderer chosen in Step 2 is read, never
    re-decided, by every step after it.
    """
    expected_asset_ids = [str(s["id"]) for s in suspects]

    characters = generate_characters(victim, suspects)          # Step 1
    solution = select_murderer(characters)                      # Step 2 - locked here
    murder = build_murder(characters, solution)                 # Step 3
    investigation = generate_investigation_content(              # Step 4
        characters, solution, murder, puzzle_categories
    )
    explanation = generate_final_explanation(                   # Step 5
        characters, solution, murder, investigation
    )

    case_json = _assemble_case_json(characters, solution, murder, investigation, explanation)
    _final_consistency_check(case_json, puzzle_categories, expected_asset_ids)  # Final Validation

    # Attach the already-selected portrait data so the UI never needs to
    # re-derive which image belongs to which generated character.
    suspect_image_by_id = {str(s["id"]): s["image_url"] for s in suspects}
    case_json["victim"]["image_url"] = victim["image_url"]
    for suspect_entry in case_json["suspects"]:
        case_json_id = str(suspect_entry.get("asset_id", ""))
        suspect_entry["image_url"] = suspect_image_by_id.get(case_json_id)

    return HiddenMemory(case_json=case_json)


print("✅ Case generator ready (locked-murderer pipeline: characters -> murderer lock -> murder -> investigation -> explanation -> validation).")


✅ Case generator ready (locked-murderer pipeline: characters -> murderer lock -> murder -> investigation -> explanation -> validation).


## 9 · Crime Scene Generator

Converts the structured `crime_scene` object from the hidden case into
one cinematic prompt string, then renders it once with Pollinations.
Suspect and victim portraits are never touched here.

In [9]:
def _build_crime_scene_prompt(crime_scene: dict) -> str:
    """Return the cinematic image-generation prompt for the crime scene.

    FIX (Problem 6): the case-generation prompt explicitly instructs the
    LLM to write a complete, ready-to-use "image_prompt" field, and
    _validate_case_json now guarantees it is present and non-empty. Use it
    directly instead of silently reconstructing a prompt from other
    fields — that reconstruction was also referencing a field name
    ("environmental_details") that doesn't exist anywhere in the case
    schema (the real field is "environmental_storytelling"), so it always
    rendered as an empty "Environmental details: ." fragment.

    The manual reconstruction is kept as a fallback only, for older/cached
    case JSON that predates the image_prompt field.
    """
    image_prompt = (crime_scene.get("image_prompt") or "").strip()
    if image_prompt:
        return image_prompt

    objects = ", ".join(crime_scene.get("objects", []))
    clues = ", ".join(crime_scene.get("important_clues", []))

    return textwrap.dedent(f"""
        Cinematic noir crime scene photograph.
        Location: {crime_scene.get('location', '')}.
        Lighting: {crime_scene.get('lighting', '')}.
        Mood: {crime_scene.get('mood', '')}.
        Camera angle: {crime_scene.get('camera_angle', '')}.
        Victim position (implied, no graphic gore): {crime_scene.get('victim_position', '')}.
        Notable objects in frame: {objects}.
        Visible clues: {clues}.
        Environmental details: {crime_scene.get('environmental_storytelling', '')}.
        Color palette: {crime_scene.get('color_palette', '')}.
        Art style: {crime_scene.get('art_style', '')}.
        High detail, atmospheric, detective noir illustration.
    """).strip().replace("\n", " ")


def generate_crime_scene(hidden: HiddenMemory) -> Path:
    """Build the prompt from the hidden case and render the crime scene
    exactly once. Returns the local file path of the generated image.
    """
    crime_scene = hidden.case_json["crime_scene"]
    prompt = _build_crime_scene_prompt(crime_scene)
    if not prompt.strip():
        # Defensive: _validate_case_json should already have caught a
        # missing image_prompt at generation time, but a case object
        # could in principle reach here from somewhere else (a cached
        # file, a manual edit), so fail loudly and specifically rather
        # than silently calling the image API with an empty prompt.
        raise ValueError(
            "Cannot generate the crime scene image: crime_scene has no "
            "usable prompt (image_prompt is empty and there isn't enough "
            "structured detail to reconstruct one)."
        )
    seed = CRIME_SCENE_SEED if CRIME_SCENE_SEED is not None else random.randint(1, 1_000_000)
    return make_image(prompt, file_name=CRIME_SCENE_FILENAME, seed=seed)


print("✅ Crime scene generator ready.")


✅ Crime scene generator ready.


## Backend Smoke Test

Runs the full backend pipeline once, with no UI: load assets → load
puzzles → cast characters → generate the hidden case → generate and
display the crime scene. This is the same sequence Section 14 (Launch)
will trigger from a button in later phases — running it here lets you
verify Sections 1–9 work end-to-end before any UI is built.

This is the ONLY cell in the notebook that uses `print()` for narration,
and it's temporary — it will be removed once the Frontend (Section 13)
takes over as the sole way information is revealed to the player.

In [10]:
def run_backend_smoke_test():
    global hidden_memory, investigation_memory, loaded_puzzles, crime_scene_path

    print("Loading suspects and victims from GitHub...")
    all_suspects = load_characters("suspects")
    all_victims = load_characters("victims")

    print("Loading puzzle library from GitHub...")
    loaded_puzzles = load_all_puzzles()
    puzzle_categories = usable_puzzle_categories(loaded_puzzles)
    print(f"  Puzzle categories found: {list(loaded_puzzles.keys())}")
    print(f"  Puzzle categories usable (non-empty): {puzzle_categories}")

    print("Casting victim + 3 suspects...")
    victim, suspects, cast_info  = select_cast(all_victims, all_suspects)

    print("Generating hidden case with the LLM...")
    hidden_memory = generate_case(victim, suspects, puzzle_categories)
    investigation_memory = InvestigationMemory()

    print(f"Case generated: \"{hidden_memory.case_json['case_title']}\"")
    print("Generating crime scene image...")
    crime_scene_path = generate_crime_scene(hidden_memory)

    print("✅ Backend smoke test complete. Hidden solution is stored and NOT printed.")

# Uncomment to run manually:
# run_backend_smoke_test()

---
# Phase 3 — Puzzle Engine & Verdict System

Fills in the two architecture sections Phase 2 left as placeholders:

* **Section 10 — Puzzle Engine**: reads the `puzzle_category` the LLM assigned to each evidence item, randomly selects an unused puzzle JSON from that category (loaded back in Section 3), and checks the player's submitted answer against it. A puzzle is locked to its evidence item the moment it's first assigned — wrong answers don't re-roll it — and once solved it's retired for the rest of the investigation.
* **Section 12 — Verdict Engine**: scores the player's accusation against `HiddenMemory` (exact match on the killer, keyword-overlap grading on motive/explanation since those are free text) and assigns a detective rank.

Both engines are wired into the Evidence and Verdict panels below, and Phase 4 adds motion polish (fade-in panels, correct/wrong flashes) plus a **Start a New Investigation** flow so the notebook never needs to be re-run to play again.

## 10 · Puzzle Engine

In [11]:
# --- Puzzle lookup -----------------------------------------------------------
# All puzzles were already loaded from GitHub in Section 3 (`load_all_puzzles`)
# into `loaded_puzzles: {category: [puzzle_dict, ...]}`. Nothing here ever
# talks to the LLM or GitHub again — the puzzle engine is pure Python logic
# over data that's already local.

def usable_puzzle_categories(all_loaded_puzzles: dict[str, list[dict]]) -> list[str]:
    """Categories that actually have at least one puzzle file in them.

    FIX (Problem 8): a category folder can exist on GitHub with zero JSON
    files inside it (or all of them fail to parse). The evidence panel
    already has a graceful fallback for that case (auto-unlocking the
    evidence instead of crashing), but it's better to never hand the LLM
    an empty category to assign evidence to in the first place — this is
    what both case-generation call sites should pass in as
    `puzzle_categories`, instead of every discovered folder name.
    """
    return [category for category, puzzles in all_loaded_puzzles.items() if puzzles]


def get_puzzles_for_category(category: str) -> list[dict]:
    """All puzzle JSON dicts discovered under puzzles/<category>/ on GitHub."""
    if not loaded_puzzles:
        return []
    return loaded_puzzles.get(category, [])


def _pick_random_puzzle(category: str) -> Optional[dict]:
    """Randomly choose one puzzle from a category, preferring puzzles that
    haven't been solved yet this investigation (so repeat evidence items
    don't recycle the exact same puzzle). Returns None only if the category
    itself has no JSON files in the repo at all.
    """
    all_in_category = get_puzzles_for_category(category)
    if not all_in_category:
        return None

    fresh = [p for p in all_in_category if p["source_file"] not in investigation_memory.solved_puzzles]
    pool = fresh if fresh else all_in_category  # every puzzle solved -> allow reuse rather than dead-end
    return random.choice(pool)


def get_or_assign_puzzle(evidence_title: str, category: str) -> Optional[dict]:
    """Return the puzzle already locked in for this evidence item, or pick
    and store a fresh random one. Stays stable across wrong-answer retries —
    the player keeps fighting the same puzzle rather than getting re-rolled.
    """
    assigned = investigation_memory.get_assigned_puzzle(evidence_title)
    if assigned is not None:
        return assigned

    puzzle = _pick_random_puzzle(category)
    if puzzle is not None:
        investigation_memory.assign_puzzle(evidence_title, puzzle)
    return puzzle


# --- Answer checking -----------------------------------------------------------
def _normalize_answer(text: str) -> str:
    """Fold away everything that shouldn't matter to whether an answer is
    "right", so the player is never penalized for formatting.

    FIX (Problem 6): validation used to expect something close to an
    exact sentence match, which is a terrible player experience — a
    missing comma or a capital letter would fail a correct answer. This
    now, in order:
      * trims leading/trailing whitespace
      * lowercases, so capitalization is ignored entirely
      * turns every run of punctuation (commas, periods, hyphens,
        apostrophes, exclamation points, etc.) into a single space,
        rather than deleting it — so punctuation is never required to
        match, but two words that were only ever separated by
        punctuation (e.g. "A-C-B-D" / "A, C, B, D") don't get
        accidentally fused into one token
      * collapses any resulting run of whitespace (extra spaces,
        double spaces, tabs) down to a single space
    Both the player's submission and every one of the puzzle's
    accepted_answers are run through this same function before being
    compared, so "  Meet me at MIDNIGHT!!" and "meet me at midnight"
    are treated as identical.
    """
    text = text.strip().lower()
    text = re.sub(r"[^\w\s]+", " ", text)   # any punctuation run -> a single space (never deleted outright)
    text = re.sub(r"\s+", " ", text).strip()  # collapse extra/duplicate whitespace
    return text


def _accepted_answers(puzzle: dict) -> list[str]:
    """Every string that counts as correct for this puzzle, whatever the
    JSON happens to call it.

    FIX (real schema check): the puzzle JSON files actually published in
    the GitHub puzzle repo (schema_version 1.0) ship a single `answer`
    string per puzzle -- not an `accepted_answers` list. That mismatch
    meant check_puzzle_answer was comparing against an empty list and no
    submission could ever be marked correct, regardless of what the
    player typed. `accepted_answers` is still checked first and used as
    the source of truth if a puzzle file does provide several valid
    phrasings; `answer` is the fallback that matches every puzzle file
    actually seen in the repo today.
    """
    accepted = puzzle.get("accepted_answers")
    if accepted:
        return [accepted] if isinstance(accepted, str) else list(accepted)
    single = puzzle.get("answer")
    return [single] if single else []


def check_puzzle_answer(puzzle: dict, submitted: str) -> bool:
    """Case-insensitive, punctuation-insensitive match against every
    string _accepted_answers() returns for this puzzle.

    Matching itself is unchanged: the submitted text and every candidate
    answer are both run through the forgiving _normalize_answer() above
    (so capitalization, extra whitespace, and punctuation are ignored),
    and a match against any one candidate counts as correct. `solution`
    is never read or compared against here — it exists purely for the
    reveal-after-failure UI, not for grading.
    """
    submitted_normalized = _normalize_answer(submitted)
    return any(
        submitted_normalized == _normalize_answer(candidate)
        for candidate in _accepted_answers(puzzle)
    )


# Player gets MAX_PUZZLE_ATTEMPTS tries total. A wrong attempt N (N < the
# last attempt) reveals hint N, capped at however many hints the puzzle
# actually has. After the final attempt is also wrong, the solution is
# revealed automatically and the puzzle is retired for 0 points — see
# exhaust_puzzle(). This one constant drives both the UI's "Attempts
# Remaining" counter and the hint schedule, so they can never drift out
# of sync with each other.
MAX_PUZZLE_ATTEMPTS = 3
REVEAL_SOLUTION_AFTER_ATTEMPTS = MAX_PUZZLE_ATTEMPTS  # kept as an alias; existing name some UI code refers to

DEFAULT_PUZZLE_REWARD_POINTS = 10  # used only if a puzzle JSON omits "reward_points"


def solve_puzzle(evidence_title: str, puzzle: dict, submitted: str) -> bool:
    """Check the player's submission, record the attempt, and — on a
    correct answer — permanently retire the puzzle, unlock the evidence,
    and award the puzzle's full reward_points.

    FIX (reward logic): a correct answer now always earns the full
    reward_points, no matter how many hints were shown along the way —
    points are withheld only in the one case where the player never
    solves it at all and the solution has to be auto-revealed after the
    final attempt (see exhaust_puzzle below), never just for having
    looked at a hint.
    """
    correct = check_puzzle_answer(puzzle, submitted)
    investigation_memory.record_puzzle_attempt(evidence_title)

    if correct:
        investigation_memory.mark_puzzle_solved(puzzle["source_file"])
        investigation_memory.unlock_evidence(evidence_title)
        points = int(puzzle.get("reward_points") or DEFAULT_PUZZLE_REWARD_POINTS)
        investigation_memory.record_puzzle_points(evidence_title, points)

    return correct


def exhaust_puzzle(evidence_title: str, puzzle: dict) -> None:
    """Called once the player's final attempt is also wrong: the puzzle
    is retired for 0 points and the evidence unlocks anyway, so a puzzle
    can never permanently block the investigation. The solution text
    itself is shown by the UI, not returned here — this only updates
    game state.
    """
    investigation_memory.mark_puzzle_solved(puzzle["source_file"])
    investigation_memory.unlock_evidence(evidence_title)
    investigation_memory.record_puzzle_points(evidence_title, 0)


print("✅ Puzzle engine ready.")

✅ Puzzle engine ready.


---
# Phase 2 — UI & Investigation Flow

Builds on Phase 1's backend. Adds:

* **Section 11 — Investigation Logic**: the suspect question system
  (max 1 question/suspect, max 3 total LLM calls) and navigation state.
* **Section 13 — Frontend**: a noir case-file styled `ipywidgets` app —
  Home Screen → crime scene reveal → Investigation Hub with tabbed
  navigation (Victim / Suspects / Evidence / Log / Verdict), one panel
  visible at a time, no vertical scrolling.
* **Section 14 — Launch**: the single cell that boots the app.

Evidence stays locked with a placeholder in this phase — the puzzle
unlock flow and Verdict Engine scoring are Phase 3. Run every cell in
order once, then run the Launch cell.

## 11 · Investigation Logic

Navigation state (which screen/panel/suspect is active) and the
question-asking system live here. This is the only place that calls
`call_llm` for suspect dialogue, so the "3 total calls" rule is
enforced in exactly one function.

In [12]:
# --- Navigation state --------------------------------------------------------
# A tiny state object instead of scattered globals, so Section 13 always
# reads/writes navigation through one place.
@dataclass
class NavState:
    screen: str = "home"          # "home" | "crime_scene" | "hub"
    panel: str = "victim"          # "victim" | "suspects" | "evidence" | "log" | "verdict"
    open_suspect: Optional[str] = None   # suspect name currently in dossier view
    open_evidence: Optional[str] = None  # evidence title currently in puzzle/detail view

nav_state = NavState()


def _find_suspect(name: str) -> dict:
    for suspect in hidden_memory.suspects:
        if suspect["name"] == name:
            return suspect
    raise ValueError(f"Unknown suspect: {name}")


# NOTE: live suspect questioning (DIALOGUE_SYSTEM_PROMPT / ask_suspect_question)
# has been removed entirely. Every suspect's interrogation record is now
# pre-written at case-generation time and stored on the suspect object as
# "investigation_notes" (see Section 8) -- the Suspects panel (Section 13e)
# reads it directly with zero additional LLM calls during gameplay.

print("✅ Investigation logic ready.")


✅ Investigation logic ready.


## 12 · Verdict Engine

In [13]:
# --- Verdict -------------------------------------------------------------
# FIX (Problem 6-9): the verdict used to ask for murderer + free-text
# motive + free-text explanation, score the motive/explanation against the
# hidden case with a keyword-overlap heuristic, and then make an extra
# LLM call to narrate the result. All of that is gone: the player now only
# names the murderer, and the entire reveal (motive, how it happened,
# supporting evidence) is the "final_solution" object that was written
# once, during case generation (Section 8) -- there is no LLM call here at
# all, correct or not.

def submit_verdict(submitted_killer: str) -> dict:
    """Score the single murderer guess against HiddenMemory and store the
    result on InvestigationMemory so the reveal is stable if the player
    navigates away from the Verdict panel and back. No LLM call.
    """
    killer_correct = submitted_killer.strip().lower() == hidden_memory.killer.strip().lower()
    result = {
        "submitted_killer": submitted_killer,
        "killer_correct": killer_correct,
    }
    investigation_memory.verdict = result
    return result


print("✅ Verdict engine ready.")


✅ Verdict engine ready.


## 13 · Frontend

### 13a · Design System

A noir case-file aesthetic, not a generic light dashboard: near-black
"desk" background, worn-leather borders, aged-paper panels, and a single
gold accent reserved for the case stamp and active tab — everything
else stays quiet so that accent reads as deliberate.

**Tokens**

| Role | Value |
|---|---|
| Background (desk) | `#0d0b08` |
| Panel (paper) | `#1c1710` |
| Card (folder) | `#241d13` |
| Border (leather) | `#3d3220` |
| Text (ink) | `#e8dfc8` |
| Text muted | `#9c9174` |
| Accent (gold stamp) | `#b8892b` |
| Danger (locked/red string) | `#7a2f2a` |
| Display face | "Special Elite" (typewriter) |
| Body face | "Crimson Pro" (serif) |
| Utility face | "IBM Plex Mono" (evidence tags, timestamps) |

**Signature element:** the Investigation Hub's navigation reads as
manila folder tabs clipped to the case file, with a rotated gold
"CASE FILE" stamp in the corner — the one bold flourish; every other
element (buttons, cards, dossiers) stays flat and restrained.

The whole app is a single fixed-size "desk" (`960×620px`) so it plays
like a desktop application panel, not a scrolling notebook page.

In [14]:
VELVET_CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Special+Elite&family=Crimson+Pro:wght@400;600&family=IBM+Plex+Mono:wght@400;500&display=swap');

.velvet-app, .velvet-app * {
    box-sizing: border-box;
    font-family: 'Crimson Pro', serif;
}
.velvet-app {
    width: 960px;
    height: 620px;
    background: #0d0b08;
    border: 1px solid #3d3220;
    border-radius: 4px;
    color: #e8dfc8;
    position: relative;
    overflow: hidden;
    padding: 0;
}
.velvet-stamp {
    position: absolute;
    top: 14px;
    right: 22px;
    transform: rotate(6deg);
    border: 2px solid #b8892b;
    color: #b8892b;
    font-family: 'Special Elite', monospace;
    font-size: 11px;
    letter-spacing: 2px;
    padding: 3px 10px;
    opacity: 0.85;
    pointer-events: none;
}
.velvet-title {
    font-family: 'Special Elite', monospace;
    letter-spacing: 1px;
}
.velvet-h1 { font-size: 30px; color: #e8dfc8; margin: 0 0 6px 0; }
.velvet-h2 { font-size: 18px; color: #b8892b; margin: 0 0 10px 0; }
.velvet-muted { color: #9c9174; font-size: 14px; }
.velvet-mono { font-family: 'IBM Plex Mono', monospace; font-size: 12px; color: #9c9174; letter-spacing: 0.5px; }

.velvet-home {
    background:
        radial-gradient(ellipse at top left, rgba(184,137,43,0.06), transparent 55%),
        #0d0b08;
}
.velvet-tagline { font-size: 16px; color: #c9bd9c; max-width: 520px; margin: 14px 0 30px 0; line-height: 1.5; }

.velvet-card {
    background: #241d13;
    border: 1px solid #3d3220;
    border-radius: 3px;
    padding: 16px 18px;
}

.velvet-navbar {
    display: flex;
    align-items: flex-end;
    gap: 4px;
    padding: 18px 24px 0 24px;
    border-bottom: 1px solid #3d3220;
    background: #14100a;
}
.velvet-tab {
    font-family: 'Special Elite', monospace;
    font-size: 13px;
    letter-spacing: 1px;
    color: #9c9174;
    background: #1c1710;
    border: 1px solid #3d3220;
    border-bottom: none;
    border-radius: 6px 6px 0 0;
    padding: 9px 18px;
    cursor: pointer;
    transition: background 0.18s ease, color 0.18s ease;
}
.velvet-tab.active,
.velvet-tab-active-placeholder {
    color: #0d0b08 !important;
    background: #b8892b !important;
    border-color: #b8892b !important;
}

.velvet-panel-body {
    padding: 22px 28px !important;
    height: 462px !important;
    min-height: 0 !important;
    overflow-y: auto !important;
    overflow-x: hidden !important;
}

.velvet-dossier-photo {
    width: 120px;
    height: 120px;
    object-fit: cover;
    border: 2px solid #3d3220;
    border-radius: 3px;
}

.velvet-locked {
    color: #7a2f2a;
    font-family: 'IBM Plex Mono', monospace;
    font-size: 11px;
    letter-spacing: 1px;
    border: 1px dashed #7a2f2a;
    display: inline-block;
    padding: 2px 8px;
    border-radius: 2px;
}

.velvet-app .widget-button {
    font-family: 'Special Elite', monospace !important;
    letter-spacing: 1px;
    background: #b8892b !important;
    color: #0d0b08 !important;
    border: none !important;
    border-radius: 3px !important;
    transition: background 0.15s ease, opacity 0.15s ease, transform 0.1s ease;
}
.velvet-app .widget-button:hover:not(:disabled) {
    background: #cf9c34 !important;
}
.velvet-app .widget-button:active:not(:disabled) {
    transform: translateY(1px);
}
.velvet-app .widget-button:disabled {
    opacity: 0.45 !important;
    cursor: default !important;
}
.velvet-app .velvet-secondary-btn .widget-button {
    background: transparent !important;
    color: #e8dfc8 !important;
    border: 1px solid #3d3220 !important;
}

/* --- Phase 4: motion polish -------------------------------------------- */
/* Applied to freshly-rendered content (panels, screens, evidence/verdict
   detail views). Because Output widgets are cleared and refilled with new
   DOM nodes on every render, this animation replays automatically each
   time a `.velvet-fade-in` element is (re)displayed — no JS required. */
.velvet-fade-in {
    animation: velvetFadeIn 0.32s ease;
}
@keyframes velvetFadeIn {
    from { opacity: 0; transform: translateY(5px); }
    to   { opacity: 1; transform: translateY(0); }
}

.velvet-flash-correct {
    color: #6fae7a;
    font-size: 13px;
    letter-spacing: 0.5px;
}
.velvet-flash-wrong {
    color: #c96a52;
    font-size: 13px;
    letter-spacing: 0.5px;
    animation: velvetShake 0.28s ease;
}
@keyframes velvetShake {
    0%, 100% { transform: translateX(0); }
    25%      { transform: translateX(-4px); }
    75%      { transform: translateX(4px); }
}

.velvet-rank-badge {
    display: inline-block;
    font-family: 'Special Elite', monospace;
    letter-spacing: 1px;
    font-size: 14px;
    color: #0d0b08;
    background: #b8892b;
    padding: 6px 14px;
    border-radius: 3px;
    margin-top: 6px;
}

/* FIX (Task 2 -- loading notice): shown near the top of the page while
   ipywidgets' comm layer is still spinning up the actual interactive
   controls, which can visibly lag behind the page load in a notebook
   environment. Kept intentionally understated (muted gold, not an error
   color) since this is expected, informational, and temporary. */
.velvet-loading-notice {
    width: 960px;
    box-sizing: border-box;
    margin: 0 0 10px 0;
    padding: 10px 16px;
    background: #1c1710;
    border: 1px solid #3d3220;
    border-left: 3px solid #b8892b;
    border-radius: 3px;
    color: #c9bd9c;
    font-family: 'IBM Plex Mono', monospace;
    font-size: 12px;
    line-height: 1.6;
}
</style>
"""

style_injector = widgets.HTML(VELVET_CSS)
print("✅ Design system loaded.")


✅ Design system loaded.


### 13b · Home Screen

The player's first view. One action: **Begin Investigation**. Nothing is
printed while the case generates — a status line inside the same styled
panel updates instead.

In [15]:
home_status_html = widgets.HTML("")

begin_button = widgets.Button(description="Begin Investigation")
begin_button.layout = widgets.Layout(margin="24px 0 0 0")

home_intro_html = widgets.HTML("""
    <div class="velvet-mono">VELVET ENVELOPE — CASE INTAKE</div>
    <div class="velvet-h1 velvet-title">A body. A room. Three liars.</div>
    <div class="velvet-tagline">
      Every case is generated fresh: a new victim, three suspects,
      and one hidden killer. Review the evidence and interrogation
      notes, then name your murderer.
    </div>
""")

# The home screen is one VBox that IS the "velvet-home" flex container,
# sized and centered entirely through ipywidgets' own Layout (not through a
# nested CSS div claiming height:100% of an otherwise auto-sized ancestor).
# This is what the -70px negative-margin version was fighting around.
home_container = widgets.VBox(
    [home_intro_html, begin_button, home_status_html],
    layout=widgets.Layout(
        height="100%",
        width="100%",
        padding="48px 60px",
        justify_content="center",
        align_items="flex-start",
    ),
)
home_container.add_class("velvet-home")


def _set_home_status(message: str, tone: str = "muted"):
    color = "#9c9174" if tone == "muted" else "#b8892b"
    home_status_html.value = (
        f'<div style="margin-top:14px; color:{color}; '
        f"font-family:'IBM Plex Mono',monospace; font-size:13px;\">{message}</div>"
    )


print("✅ Home screen built.")


✅ Home screen built.


### 13c · Investigation Hub Shell

The folder-tab navigation bar plus a single panel-body area. Only one
panel's content is ever rendered into `panel_body` at a time — this is
what keeps the app on one screen with no vertical scrolling between
sections.

In [16]:
PANEL_LABELS = {
    "crime_scene": "CRIME SCENE",
    "victim": "VICTIM",
    "suspects": "SUSPECTS",
    "evidence": "EVIDENCE",
    "log": "INVESTIGATION LOG",
    "verdict": "VERDICT",
}

nav_buttons: dict[str, widgets.Button] = {}
for key, label in PANEL_LABELS.items():
    button = widgets.Button(description=label)
    button.add_class("velvet-tab")
    nav_buttons[key] = button

navbar = widgets.HBox([nav_buttons[k] for k in PANEL_LABELS], layout=widgets.Layout(gap="4px"))
navbar_wrapper = widgets.HTML('<div class="velvet-navbar" style="height:0;padding-top:0;border:none;"></div>')

panel_body = widgets.Output(layout=widgets.Layout(height="462px", overflow_y="auto", padding="22px 28px"))
# FIX (submit buttons appearing to vanish): .velvet-panel-body already
# existed in VELVET_CSS but was never actually applied to this widget,
# so scrolling relied only on ipywidgets' own (not always reliable,
# especially in Colab) translation of the Layout above into CSS.
# Wiring the real class in guarantees the panel scrolls and whatever
# is at the bottom of a tall panel (Submit Verdict, Ask Question,
# Submit puzzle answer) is always reachable.
panel_body.add_class("velvet-panel-body")

hub_container = widgets.VBox([navbar, panel_body])


# FIX (appear-then-disappear flicker): every navigation action in the
# app -- switching tabs, opening/closing an evidence item or suspect
# dossier, submitting a verdict -- funnels through render_current_panel(),
# and each call does a full clear_output(wait=True) + redraw. If a click
# fires its handler twice (widget-sync latency occasionally double-fires
# a single click, or the player double-clicks because nothing looked like
# it registered the first time), two full redraws get queued back to
# back: the browser shows the first render's content, the second call's
# clear_output wipes it before the second render is ready, then the
# second render appears -- the panel visibly pops in, vanishes, then
# pops in again. Debouncing repeat calls that land within
# PANEL_RENDER_DEBOUNCE_SECONDS of the previous one stops that duplicate
# clear+redraw pair from ever being queued.
_LAST_PANEL_RENDER_TS = 0.0
PANEL_RENDER_DEBOUNCE_SECONDS = 0.35


def render_current_panel():
    """Dispatch to the render function for whichever panel is active."""
    global _LAST_PANEL_RENDER_TS
    now = time.time()
    if now - _LAST_PANEL_RENDER_TS < PANEL_RENDER_DEBOUNCE_SECONDS:
        return  # a duplicate/near-simultaneous call -- ignore it, don't re-render
    _LAST_PANEL_RENDER_TS = now

    panel_body.clear_output(wait=True)
    for key, button in nav_buttons.items():
        if key == nav_state.panel:
            button.remove_class("velvet-tab")
            button.add_class("velvet-tab-active-placeholder")  # visual state, styled below
        else:
            button.remove_class("velvet-tab-active-placeholder")
            button.add_class("velvet-tab")
    with panel_body:
        RENDERERS[nav_state.panel]()


def switch_panel(panel_key: str):
    nav_state.panel = panel_key
    nav_state.open_suspect = None
    render_current_panel()


def _make_nav_handler(panel_key):
    def _handler(_button):
        switch_panel(panel_key)
    return _handler


for key, button in nav_buttons.items():
    button.on_click(_make_nav_handler(key))

print("✅ Hub shell built.")

✅ Hub shell built.


### 13c-ii · Crime Scene Panel

In [17]:
# FIX (Problem 2 & 12): "crime_scene" is now one of the tabs in
# PANEL_LABELS (Section 13c), but RENDERERS never had a matching entry —
# clicking the tab raised KeyError: 'crime_scene'. This adds that
# renderer and, per Problem 12, it never regenerates the image: the
# crime scene is created exactly once in _on_begin (Section 14) and
# saved to `crime_scene_path`; this just redisplays that same file from
# disk every time the tab is opened.
def render_crime_scene_panel():
    display(widgets.HTML('<div class="velvet-mono" style="margin-bottom:14px;">CRIME SCENE</div>'))

    if crime_scene_path is None or not Path(crime_scene_path).exists():
        display(widgets.HTML(
            '<div class="velvet-locked">No crime scene image is available for this investigation.</div>'
        ))
        return

    display(Image(filename=str(crime_scene_path), width=700))

    if hidden_memory is not None:
        crime_scene = hidden_memory.case_json.get("crime_scene", {})
        location = crime_scene.get("location", "")
        mood = crime_scene.get("mood", "")
        caption = " — ".join(part for part in (location, mood) if part)
        if caption:
            display(widgets.HTML(f'<div class="velvet-muted" style="margin-top:12px;">{caption}</div>'))

        # FIX: this page used to show only the image and a one-line
        # location/mood caption, with no sense of what actually happened.
        # case_json["summary"] and ["timeline"] already exist (the case
        # generator always writes them) and are player-safe -- neither
        # names the killer nor states the hidden solution -- so surface
        # them here as the case's opening narrative.
        summary = hidden_memory.case_json.get("summary", "")
        timeline = hidden_memory.case_json.get("timeline", "")

        if summary or timeline:
            summary_html = (
                f'<div style="margin-top:10px; line-height:1.6;">{summary}</div>'
                if summary else ""
            )
            timeline_html = (
                f'<div class="velvet-muted" style="margin-top:12px;">'
                f'<strong>Timeline:</strong> {timeline}</div>'
                if timeline else ""
            )
            display(widgets.HTML(f"""
                <div class="velvet-card velvet-fade-in" style="margin-top:16px;">
                  <div class="velvet-mono">THE STORY SO FAR</div>
                  {summary_html}
                  {timeline_html}
                </div>
            """))


print("✅ Crime scene panel ready.")

✅ Crime scene panel ready.


### 13d · Victim Panel

Portrait, generated identity, background, and timeline — read from
`HiddenMemory`, but only the player-safe fields (never the solution).

In [18]:
# FIX (Problem 2): the Victim page used to lead with a locked
# "BACKGROUND" section the player had to unlock evidence to even see,
# which left the very first tab almost empty. This page is now the
# briefing the detective is handed before the investigation starts:
# an always-visible "CASE SUMMARY" card with the facts already known
# going in (time of death, location, how the body was found, the
# apparent cause of death, who responded first, and the player's
# objective). None of this is secret and none of it is the solution —
# it comes from the new `briefing` object the case generator writes
# (Section 8), never from `hidden_solution`, `killer`, or `weapon`.
#
# The deeper, genuinely spoiler-heavy overview (the old locked "CASE
# SUMMARY" built from `case_json['summary']`) still exists and is
# still gated behind evidence unlocks — it's just relabeled "FULL CASE
# FILE" below so it's no longer confused with the new briefing card.
VICTIM_REVEAL_TIERS = [
    # (evidence_unlocked_required, [(section_label, field_key), ...])
    (2, [("DAILY ROUTINE", "daily_routine"), ("RECENT EVENTS", "recent_events")]),
    (3, [("TIMELINE", "timeline")]),
]


def render_victim_panel():
    victim = hidden_memory.case_json["victim"]
    briefing = hidden_memory.case_json["briefing"]
    location = hidden_memory.case_json.get("location", "")
    unlocked_count = len(investigation_memory.unlocked_evidence)

    display(widgets.HTML(f"""
        <div style="display:flex; gap:24px;">
          <img class="velvet-dossier-photo" src="{victim['image_url']}" />
          <div>
            <div class="velvet-h2 velvet-title">{victim['name']}</div>
            <div class="velvet-muted" style="margin-bottom:10px;">{victim['occupation']}</div>
            <div style="line-height:1.6; max-width:560px;">{victim['public_reputation']}</div>
          </div>
        </div>
    """))

    briefing_rows = [
        ("Estimated time of death", briefing.get("estimated_time_of_death", "")),
        ("Location", location),
        ("Victim's occupation", victim.get("occupation", "")),
        ("How the body was discovered", briefing.get("how_body_was_discovered", "")),
        ("Apparent cause of death", briefing.get("apparent_cause_of_death", "")),
        ("First responding officer", briefing.get("first_responding_officer", "")),
        ("Current objective", (
            f"Interview the suspects, examine the evidence, and determine "
            f"who killed {victim['name']} — and why."
        )),
    ]
    rows_html = "".join(
        f"""
        <div style="display:flex; gap:10px; padding:6px 0; border-bottom:1px solid rgba(255,255,255,0.08);">
          <div class="velvet-mono" style="min-width:220px; opacity:0.75;">{label.upper()}</div>
          <div style="line-height:1.5;">{value}</div>
        </div>
        """
        for label, value in briefing_rows
    )
    display(widgets.HTML(f"""
        <div class="velvet-card velvet-fade-in" style="margin-top:16px;">
          <div class="velvet-mono" style="margin-bottom:10px;">CASE SUMMARY</div>
          {rows_html}
        </div>
    """))

    for required_unlocks, sections in VICTIM_REVEAL_TIERS:
        if unlocked_count >= required_unlocks:
            for label, field_key in sections:
                display(widgets.HTML(f"""
                    <div class="velvet-card velvet-fade-in" style="margin-top:16px;">
                      <div class="velvet-mono" style="margin-bottom:6px;">{label}</div>
                      <div style="line-height:1.6;">{victim.get(field_key, '')}</div>
                    </div>
                """))
        else:
            remaining = required_unlocks - unlocked_count
            labels = " / ".join(label for label, _ in sections)
            display(widgets.HTML(f"""
                <div class="velvet-card" style="margin-top:16px; opacity:0.85;">
                  <div class="velvet-mono" style="margin-bottom:6px;">{labels}</div>
                  <div class="velvet-locked">
                    LOCKED — unlock {remaining} more piece{'s' if remaining != 1 else ''}
                    of evidence to reveal this
                  </div>
                </div>
            """))
            break  # later tiers stay hidden until this one is reached too

    # The full case overview is the single most spoiler-heavy piece of
    # victim text, so it stays gated behind the same final tier as the
    # timeline, and is labeled separately from the briefing above.
    if unlocked_count >= VICTIM_REVEAL_TIERS[-1][0]:
        display(widgets.HTML(f"""
            <div class="velvet-card velvet-fade-in" style="margin-top:16px;">
              <div class="velvet-mono" style="margin-bottom:6px;">FULL CASE FILE</div>
              <div style="line-height:1.6;">{hidden_memory.case_json['summary']}</div>
            </div>
        """))

print("✅ Victim panel ready.")


✅ Victim panel ready.


### 13e · Suspects Panel

A roster of three buttons opens a dossier for the selected suspect,
including the one-question dialogue box. Question limits are enforced
by `ask_suspect_question` in Section 11 — this section only handles
display and wiring.

In [19]:
def render_suspects_panel():
    if nav_state.open_suspect is None:
        _render_suspect_roster()
    else:
        _render_suspect_dossier(nav_state.open_suspect)


def _render_suspect_roster():
    roster_buttons = []
    for suspect in hidden_memory.suspects:
        button = widgets.Button(description=suspect["name"])
        button.layout = widgets.Layout(width="220px", height="42px", margin="0 12px 12px 0")

        def _open(_btn, name=suspect["name"]):
            nav_state.open_suspect = name
            render_current_panel()

        button.on_click(_open)
        roster_buttons.append(button)

    display(widgets.HTML('<div class="velvet-mono" style="margin-bottom:14px;">THREE SUSPECTS — SELECT ONE TO REVIEW</div>'))
    display(widgets.HBox(roster_buttons))


# FIX (Problem 6): the dossier used to show just one paragraph
# (personality) plus the initial statement and claimed alibi. The case
# generator already writes background, known_reputation,
# recent_interaction, habit, quirk, usual_clothing, noticed_by_others,
# possible_secret, and known_skills for every suspect (validated in
# Section 8) — none of it was ever displayed. It's now laid out in clean,
# labeled sections instead of being buried in one paragraph.
def _render_suspect_dossier(name: str):
    suspect = _find_suspect(name)

    back_button = widgets.Button(description="< Back to roster")
    back_button.add_class("velvet-secondary-btn")

    def _back(_btn):
        nav_state.open_suspect = None
        render_current_panel()

    back_button.on_click(_back)
    display(back_button)

    known_skills = suspect.get("known_skills") or []
    skills_line = ", ".join(str(s) for s in known_skills) if known_skills else "—"

    # FIX (question box missing): every field below used to be read with
    # suspect['field'] direct indexing. If any single field was ever
    # absent or a case_json didn't validate perfectly for it, that raised
    # a bare KeyError partway through this function -- which aborted the
    # whole render *before* reaching the section below. Every field is now
    # read with .get() and a safe fallback, and the whole display is
    # wrapped in try/except so a rendering problem can degrade the dossier
    # gracefully instead of taking the rest of the page down with it.
    try:
        display(widgets.HTML(f"""
            <div style="display:flex; gap:24px; margin-top:14px;">
              <img src="{suspect.get('image_url', '')}"
                    style="height:360px;" />
              <div>
                <div class="velvet-h2 velvet-title">{suspect.get('name', name)}</div>
                <div class="velvet-muted">{suspect.get('occupation', '')} &middot; {suspect.get('relationship', '')}</div>
                <div style="margin-top:8px; line-height:1.6; max-width:560px;">{suspect.get('personality', '')}</div>
              </div>
            </div>

            <div class="velvet-card" style="margin-top:18px;">
              <div class="velvet-mono" style="margin-bottom:6px;">INITIAL STATEMENT</div>
              <div style="line-height:1.6; font-style:italic;">&ldquo;{suspect.get('initial_statement', '')}&rdquo;</div>
            </div>

            <div style="display:flex; gap:16px; margin-top:16px;">
              <div class="velvet-card" style="flex:1;">
                <div class="velvet-mono">CLAIMED LOCATION</div>
                <div style="margin-top:4px;">{suspect.get('claimed_location', '')}</div>
              </div>
              <div class="velvet-card" style="flex:1;">
                <div class="velvet-mono">CLAIMED ACTIVITY</div>
                <div style="margin-top:4px;">{suspect.get('claimed_activity', '')}</div>
              </div>
            </div>

            <div class="velvet-card" style="margin-top:16px;">
              <div class="velvet-mono" style="margin-bottom:6px;">BACKGROUND</div>
              <div style="line-height:1.6;">{suspect.get('background', '')}</div>
            </div>

            <div class="velvet-card" style="margin-top:16px;">
              <div class="velvet-mono" style="margin-bottom:6px;">KNOWN REPUTATION</div>
              <div style="line-height:1.6;">{suspect.get('known_reputation', '')}</div>
            </div>

            <div class="velvet-card" style="margin-top:16px;">
              <div class="velvet-mono" style="margin-bottom:6px;">RECENT INTERACTION WITH VICTIM</div>
              <div style="line-height:1.6;">{suspect.get('recent_interaction', '')}</div>
            </div>

            <div style="display:flex; gap:16px; margin-top:16px;">
              <div class="velvet-card" style="flex:1;">
                <div class="velvet-mono">HABIT</div>
                <div style="margin-top:4px;">{suspect.get('habit', '')}</div>
              </div>
              <div class="velvet-card" style="flex:1;">
                <div class="velvet-mono">QUIRK</div>
                <div style="margin-top:4px;">{suspect.get('quirk', '')}</div>
              </div>
              <div class="velvet-card" style="flex:1;">
                <div class="velvet-mono">USUAL CLOTHING</div>
                <div style="margin-top:4px;">{suspect.get('usual_clothing', '')}</div>
              </div>
            </div>

            <div class="velvet-card" style="margin-top:16px;">
              <div class="velvet-mono" style="margin-bottom:6px;">NOTICED BY OTHERS</div>
              <div style="line-height:1.6;">{suspect.get('noticed_by_others', '')}</div>
            </div>

            <div class="velvet-card" style="margin-top:16px;">
              <div class="velvet-mono" style="margin-bottom:6px;">POSSIBLE SECRET</div>
              <div style="line-height:1.6;">{suspect.get('possible_secret', '')}</div>
            </div>

            <div class="velvet-card" style="margin-top:16px;">
              <div class="velvet-mono" style="margin-bottom:6px;">KNOWN SKILLS</div>
              <div style="line-height:1.6;">{skills_line}</div>
            </div>
        """))
    except Exception as error:
        display(widgets.HTML(
            f'<div class="velvet-locked" style="margin-top:14px;">'
            f'Some of this suspect\'s dossier couldn\'t be displayed ({error}), '
            f'but their interrogation notes are still below.</div>'
        ))

    # Always reached now, regardless of whether the dossier above rendered
    # cleanly.
    _render_investigation_notes(suspect)


# FIX: replaces the old "Ask Your One Question" live-LLM feature entirely.
# Every suspect's full interrogation transcript (4-6 Q&A pairs) is written
# once, during case generation (Section 8), and stored on the suspect
# object as "investigation_notes". This just displays that stored data --
# there is no button, no textbox, no loading state, and no LLM call here.
# It reads like a detective's own interview notes rather than a chat log:
# plain text, no bubbles, no cards around each exchange.
# FIX (Task 5 -- fallback behavior): this used to bail out to the "no
# notes" message only when the raw list was empty, then unconditionally
# render a container built from `rows` for any non-empty list. If every
# entry in that list happened to be malformed (not a dict, or missing a
# question/answer), the filtering loop below would leave `rows` empty
# while `notes` itself was non-empty -- so this fell through past the
# early return and rendered an empty, contentless <div>, exactly the
# "blank section" the UI must never show. The fallback message is now
# shown any time there are zero *usable* notes, not zero raw notes.
INVESTIGATION_NOTES_UNAVAILABLE_TEXT = "Investigation notes unavailable for this case."


def _render_investigation_notes(suspect: dict):
    raw_notes = suspect.get("investigation_notes") or []

    display(widgets.HTML('<div class="velvet-mono" style="margin-top:22px; margin-bottom:2px;">INVESTIGATION NOTES</div>'))

    rows = []
    for note in raw_notes:
        # Defensive: validation (Section 8) guarantees every note is a
        # well-formed {"question", "answer"} dict before a case ever
        # reaches gameplay, but every other renderer in this file treats
        # LLM-shaped data as untrusted at render time too -- this just
        # keeps that pattern consistent instead of being the one place
        # that assumes the shape is always perfect.
        if not isinstance(note, dict):
            continue
        question = note.get("question", "")
        answer = note.get("answer", "")
        if not question or not answer:
            continue
        rows.append(f"""
            <div style="margin-top:14px;">
              <div class="velvet-mono" style="color:#9c9174;">Q:</div>
              <div style="line-height:1.6;">{question}</div>
              <div class="velvet-mono" style="color:#9c9174; margin-top:8px;">A:</div>
              <div style="line-height:1.6;">{answer}</div>
            </div>
            <div style="border-top:1px dashed #3d3220; margin-top:14px;"></div>
        """)

    if not rows:
        display(widgets.HTML(
            f'<div class="velvet-muted" style="margin-top:6px;">{INVESTIGATION_NOTES_UNAVAILABLE_TEXT}</div>'
        ))
        return

    display(widgets.HTML(f"""
        <div class="velvet-fade-in" style="font-family:'Crimson Pro', serif; max-width:640px;">
          {''.join(rows)}
        </div>
    """))

print("✅ Suspects panel ready.")


✅ Suspects panel ready.


### 13f · Evidence Panel

Shows every evidence item as locked. Clicking to solve its puzzle and
unlock it is the Puzzle Engine's job — Phase 3.

In [20]:
def render_evidence_panel():
    if nav_state.open_evidence is None:
        _render_evidence_locker()
    else:
        _render_evidence_detail(nav_state.open_evidence)


def _render_evidence_locker():
    """The list view: every evidence item, locked or unlocked."""
    display(widgets.HTML('<div class="velvet-mono" style="margin-bottom:14px;">EVIDENCE LOCKER</div>'))

    for evidence in hidden_memory.evidence:
        title = evidence["title"]
        unlocked = title in investigation_memory.unlocked_evidence
        row = widgets.Output()

        with row:
            if unlocked:
                display(widgets.HTML(f"""
                    <div class="velvet-card velvet-fade-in" style="margin-bottom:12px;">
                      <div class="velvet-h2" style="font-size:15px;">{title}</div>
                      <div style="margin-top:6px; line-height:1.6;">{evidence['description']}</div>
                      <div class="velvet-muted" style="margin-top:8px;">Relevance: {evidence['relevance']}</div>
                    </div>
                """))
            else:
                display(widgets.HTML(f"""
                    <div class="velvet-card velvet-fade-in" style="margin-bottom:8px; opacity:0.85;">
                      <div class="velvet-h2" style="font-size:15px;">{title}</div>
                      <div class="velvet-locked" style="margin-top:6px;">LOCKED — {evidence['puzzle_category'].upper()} PUZZLE</div>
                    </div>
                """))
                open_button = widgets.Button(description="Investigate")
                open_button.layout = widgets.Layout(width="150px", margin="0 0 14px 0")

                def _open(_btn, evidence_title=title):
                    nav_state.open_evidence = evidence_title
                    render_current_panel()

                open_button.on_click(_open)
                display(open_button)

        display(row)


def _render_evidence_detail(title: str):
    """The single-item view: either the unlocked detail, or the puzzle
    that stands between the player and unlocking it.
    """
    evidence = next(e for e in hidden_memory.evidence if e["title"] == title)
    unlocked = title in investigation_memory.unlocked_evidence

    back_button = widgets.Button(description="< Back to locker")
    back_button.add_class("velvet-secondary-btn")

    def _back(_btn):
        nav_state.open_evidence = None
        render_current_panel()

    back_button.on_click(_back)
    display(back_button)

    if unlocked:
        display(widgets.HTML(f"""
            <div class="velvet-card velvet-fade-in" style="margin-top:14px;">
              <div class="velvet-h2">{title}</div>
              <div style="margin-top:8px; line-height:1.6;">{evidence['description']}</div>
              <div class="velvet-muted" style="margin-top:10px;">Relevance: {evidence['relevance']}</div>
              <div class="velvet-muted" style="margin-top:4px;">Linked to: {evidence['linked_suspect']}</div>
            </div>
        """))
        return

    category = evidence["puzzle_category"]

    # FIX (Task 5 -- fallback behavior): get_or_assign_puzzle / _render_puzzle
    # trust the puzzle dict's shape in several places (e.g. a malformed
    # puzzle JSON on GitHub missing "source_file"). Before this, an
    # unexpected failure anywhere in that path raised straight out of
    # render_current_panel() with nothing caught above it -- the player
    # would land on a half-rendered or blank panel (just the back button,
    # or a raw traceback) instead of a clear, actionable state. This is
    # the same "never leave the screen blank/half-drawn" guarantee Task 1
    # gave the answer box, applied to the whole puzzle-loading step.
    try:
        puzzle = get_or_assign_puzzle(title, category)
    except Exception:
        display(widgets.HTML("""
            <div class="velvet-card velvet-fade-in" style="margin-top:14px;">
              <div class="velvet-mono" style="color:#b8892b;">PUZZLE LOADING</div>
              <div style="margin-top:8px; line-height:1.6;">
                This puzzle couldn't be loaded right now. Go back to the
                locker and try opening this evidence again.
              </div>
            </div>
        """))
        return

    if puzzle is None:
        # Edge case: this category has zero JSON files in the GitHub repo.
        # Rather than soft-locking the player forever, unlock the evidence
        # automatically and say so plainly.
        investigation_memory.unlock_evidence(title)
        display(widgets.HTML(f"""
            <div class="velvet-card velvet-fade-in" style="margin-top:14px;">
              <div class="velvet-mono" style="color:#b8892b;">
                No {category.upper()} puzzles were found in the archive —
                this piece of evidence has been unlocked automatically.
              </div>
            </div>
        """))
        _render_evidence_detail(title)  # re-render once to show the now-unlocked detail
        return

    try:
        _render_puzzle(title, evidence, puzzle)
    except Exception:
        display(widgets.HTML("""
            <div class="velvet-card velvet-fade-in" style="margin-top:14px;">
              <div class="velvet-mono" style="color:#b8892b;">PUZZLE LOADING</div>
              <div style="margin-top:8px; line-height:1.6;">
                This puzzle couldn't be displayed right now. Go back to the
                locker and try opening this evidence again.
              </div>
            </div>
        """))


# NOTE: HINT_AFTER_ATTEMPTS now lives in the Puzzle Engine (Section 10),
# since solve_puzzle needs the same threshold to decide reward points.


def _puzzle_meta_line(puzzle: dict) -> str:
    """Build the small "TYPE · DIFFICULTY" label from whatever the puzzle
    repository actually provides.

    FIX (Problem 3): the puzzle repository was rewritten and some puzzle
    JSON files no longer include "puzzle_type" or "difficulty" at all —
    the renderer used to assume both always existed and crashed with a
    bare KeyError the moment one was missing. Every field here is now
    optional: present fields are shown, missing ones are simply skipped
    instead of crashing.

    FIX (real schema check): the puzzle files actually in the repo label
    the puzzle's kind as "category" (e.g. "cipher"), not "puzzle_type" —
    "puzzle_type" is checked first for any file that does use it, with
    "category" as the fallback so today's real puzzles show a type too.
    """
    puzzle_type = puzzle.get("puzzle_type") or puzzle.get("category")
    bits = [str(b).upper() for b in (puzzle_type, puzzle.get("difficulty")) if b]
    return " &middot; ".join(bits)


def _render_puzzle(title: str, evidence: dict, puzzle: dict):
    """Render one puzzle from its GitHub JSON, following the finalized
    puzzle schema: id, title, category, difficulty, description,
    question, acceptable_answer_format, accepted_answers, hints,
    solution_explanation, time_limit, reward_points.

    Shown to the player: category, difficulty, title, description,
    question, reward_points, and acceptable_answer_format.
    Never rendered: accepted_answers, solution_explanation, or the raw
    hints list as a whole -- those stay inside the puzzle dict and are
    only ever surfaced by check_puzzle_answer or the attempt-by-attempt
    hint/solution logic below.
    """
    meta_line = _puzzle_meta_line(puzzle)
    meta_html = f'<div class="velvet-mono">{meta_line}</div>' if meta_line else ""

    puzzle_title = puzzle.get("title") or "Puzzle"
    puzzle_description = puzzle.get("description", "")
    puzzle_question = puzzle.get("question", "")
    answer_format = puzzle.get("acceptable_answer_format") or (
        "No specific format required — just answer the question above in your own words."
    )
    reward_points = int(puzzle.get("reward_points") or DEFAULT_PUZZLE_REWARD_POINTS)

    description_html = (
        f'<div style="margin-top:8px; line-height:1.6;">{puzzle_description}</div>'
        if puzzle_description else ""
    )

    display(widgets.HTML(f"""
        <div class="velvet-card velvet-fade-in" style="margin-top:14px;">
          {meta_html}
          <div class="velvet-h2" style="margin-top:6px;">{puzzle_title}</div>
          {description_html}
          <div style="margin-top:10px; line-height:1.6;">{puzzle_question}</div>
          <div class="velvet-muted" style="margin-top:10px;">Reward: {reward_points} pts</div>
        </div>
    """))

    # --- Attempts remaining --------------------------------------------------
    attempts_output = widgets.Output()

    def _refresh_attempts():
        attempts_output.clear_output(wait=True)
        used = investigation_memory.puzzle_attempts.get(title, 0)
        remaining = max(MAX_PUZZLE_ATTEMPTS - used, 0)
        with attempts_output:
            display(widgets.HTML(f"""
                <div class="velvet-card velvet-fade-in" style="margin-top:14px;">
                  <div class="velvet-mono">ATTEMPTS REMAINING</div>
                  <div style="margin-top:4px; font-size:18px;">{remaining} / {MAX_PUZZLE_ATTEMPTS}</div>
                </div>
            """))

    display(attempts_output)
    _refresh_attempts()

    # --- Example format --------------------------------------------------------
    display(widgets.HTML(f"""
        <div class="velvet-card velvet-fade-in" style="margin-top:14px; border-style:dashed;">
          <div class="velvet-mono">EXAMPLE FORMAT</div>
          <div style="margin-top:6px; line-height:1.6;">{answer_format}</div>
        </div>
    """))

    # --- Answer input, hints, and result -------------------------------------
    answer_box = widgets.Text(placeholder="Your answer...", layout=widgets.Layout(width="420px"))
    submit_button = widgets.Button(description="Submit Answer")
    feedback_output = widgets.Output()
    hint_output = widgets.Output()

    def _get_hints() -> list[str]:
        hints = puzzle.get("hints")
        return list(hints) if hints else []

    def _refresh_hints():
        """Hints are revealed one per wrong attempt, in order, never all
        at once and never invented beyond what the puzzle actually has.
        Wrong attempt N reveals Hint N, capped at MAX_PUZZLE_ATTEMPTS - 1
        -- the final attempt reveals the solution instead of a hint,
        even if a hint for that slot exists in the JSON.
        """
        hint_output.clear_output(wait=True)
        used = investigation_memory.puzzle_attempts.get(title, 0)
        hints = _get_hints()
        hints_to_show = min(used, len(hints), MAX_PUZZLE_ATTEMPTS - 1)
        if hints_to_show <= 0:
            return
        with hint_output:
            for i in range(hints_to_show):
                display(widgets.HTML(
                    f'<div class="velvet-muted" style="margin-top:10px;">HINT {i + 1}: {hints[i]}</div>'
                ))

    def _show_exhausted():
        """The final attempt was also wrong: auto-reveal the solution,
        award 0 points, unlock the evidence anyway, and stop taking
        further input -- the puzzle never permanently blocks progress.
        """
        exhaust_puzzle(title, puzzle)
        answer_box.disabled = True
        submit_button.disabled = True
        _refresh_attempts()
        with feedback_output:
            clear_output()
            display(widgets.HTML(f"""
                <div class="velvet-flash-wrong velvet-fade-in" style="margin-top:10px;">OUT OF ATTEMPTS</div>
                <div class="velvet-card velvet-fade-in" style="margin-top:10px; border-color:#b8892b;">
                  <div class="velvet-mono" style="color:#b8892b;">SOLUTION</div>
                  <div style="margin-top:8px; line-height:1.6;">{puzzle.get("solution_explanation", "")}</div>
                  <div class="velvet-muted" style="margin-top:10px;">No reward points earned.</div>
                </div>
                <div class="velvet-card velvet-fade-in" style="margin-top:10px;">
                  <div style="line-height:1.6;">{evidence['description']}</div>
                  <div class="velvet-muted" style="margin-top:8px;">Relevance: {evidence['relevance']}</div>
                </div>
            """))

    def _submit(_btn=None):
        submitted = answer_box.value.strip()
        if not submitted:
            with feedback_output:
                clear_output()
                display(widgets.HTML(
                    '<div class="velvet-locked" style="margin-top:10px;">Type an answer before submitting.</div>'
                ))
            return

        # Instant visual confirmation the click (or Enter) landed, and a
        # guard against a second click/Enter firing a duplicate check
        # while this one is still being processed.
        submit_button.disabled = True
        submit_button.description = "Checking..."
        answer_box.disabled = True

        correct = solve_puzzle(title, puzzle, submitted)
        _refresh_attempts()

        if correct:
            earned_points = investigation_memory.puzzle_points.get(title, 0)
            with feedback_output:
                clear_output()
                display(widgets.HTML(f"""
                    <div class="velvet-flash-correct velvet-fade-in" style="margin-top:10px;">PUZZLE SOLVED</div>
                    <div class="velvet-muted" style="margin-top:2px;">Evidence successfully recovered. +{earned_points} points</div>
                    <div class="velvet-card velvet-fade-in" style="margin-top:10px;">
                      <div style="line-height:1.6;">{evidence['description']}</div>
                      <div class="velvet-muted" style="margin-top:8px;">Relevance: {evidence['relevance']}</div>
                    </div>
                """))
            answer_box.disabled = True
            submit_button.disabled = True
            return

        used = investigation_memory.puzzle_attempts.get(title, 0)
        if used >= MAX_PUZZLE_ATTEMPTS:
            _show_exhausted()
            return

        with feedback_output:
            clear_output()
            display(widgets.HTML('<div class="velvet-flash-wrong" style="margin-top:10px;">NOT QUITE. TRY AGAIN.</div>'))
        _refresh_hints()
        submit_button.description = "Submit Answer"
        answer_box.disabled = False
        submit_button.disabled = False

    # FIX (Problem: answer box sometimes never appears): this used to wire
    # answer_box.on_submit(_submit) BEFORE displaying the box/button. Text.on_submit
    # was deprecated in ipywidgets 7 and is gone in ipywidgets 8 -- calling it there
    # raises AttributeError, which aborted this function *before* the display()
    # calls below ever ran. Everything rendered up to this point (title, question,
    # attempts remaining, example format) still showed, so the page looked "almost
    # loaded" while only the input box and submit button silently never appeared --
    # exactly the reported symptom, and "sometimes" because it depends on whichever
    # ipywidgets version the kernel happens to have.
    #
    # Fix: display the controls first, so they are guaranteed to be on screen
    # regardless of what happens next, then wire interactivity afterwards inside
    # its own try/except so a binding failure can degrade (Enter-to-submit stops
    # working) instead of deleting the whole answer section.
    display(widgets.HTML('<div class="velvet-mono" style="margin-top:16px;">YOUR ANSWER</div>'))
    display(widgets.HBox([answer_box, submit_button]))
    display(hint_output)
    display(feedback_output)
    _refresh_hints()

    try:
        submit_button.on_click(_submit)
    except Exception:
        pass  # controls are already on screen either way; click just won't fire

    try:
        if hasattr(answer_box, "on_submit"):
            answer_box.on_submit(_submit)  # ipywidgets 7: real Enter-to-submit event
        else:
            # ipywidgets 8 removed Text.on_submit. continuous_update=False makes the
            # 'value' trait update on Enter/blur instead of every keystroke, so
            # observing it is the closest equivalent to "Enter submits".
            answer_box.continuous_update = False
            answer_box.observe(lambda _change: _submit(), names="value")
    except Exception:
        pass  # Submit button click still works even if Enter-to-submit can't bind


print("✅ Evidence panel ready — puzzles now unlock evidence via the Phase 3 engine.")


✅ Evidence panel ready — puzzles now unlock evidence via the Phase 3 engine.


### 13g · Investigation Log Panel

Player-visible progress only: read straight from `InvestigationMemory`,
never `HiddenMemory`.

In [21]:
def render_log_panel():
    unlocked_count = len(investigation_memory.unlocked_evidence)
    total_count = len(hidden_memory.evidence)

    # NOTE: the old "DIALOGUE CALLS USED" stat and "QUESTIONS ASKED" list
    # tracked the live questioning feature, which has been removed --
    # every suspect's interrogation record is pre-generated and always
    # fully visible on their dossier page (Section 13e), so there's
    # nothing left here to progressively unlock or count.
    display(widgets.HTML(f"""
        <div style="display:flex; gap:16px; margin-bottom:18px;">
          <div class="velvet-card" style="flex:1;">
            <div class="velvet-mono">EVIDENCE UNLOCKED</div>
            <div class="velvet-h1" style="font-size:26px; margin-top:6px;">{unlocked_count} / {total_count}</div>
          </div>
          <div class="velvet-card" style="flex:1;">
            <div class="velvet-mono">PUZZLE POINTS</div>
            <div class="velvet-h1" style="font-size:26px; margin-top:6px;">{investigation_memory.total_puzzle_points}</div>
          </div>
        </div>
    """))

print("✅ Investigation log panel ready.")


✅ Investigation log panel ready.


### 13h · Verdict Panel

Placeholder for this phase — the real accuse/score/reveal flow is the
Verdict Engine, built in Phase 3 alongside the puzzle system.

In [22]:
def render_verdict_panel():
    if investigation_memory.verdict is not None:
        _render_verdict_reveal(investigation_memory.verdict)
    else:
        _render_verdict_form()


# FIX (Problem 6): the accusation form used to also require a free-text
# motive and a free-text explanation. Both are gone -- the player now only
# names the murderer.
def _render_verdict_form():
    suspects = hidden_memory.suspects

    display(widgets.HTML('<div class="velvet-mono" style="margin-bottom:14px;">NAME THE MURDERER</div>'))

    killer_dropdown = widgets.Dropdown(
        options=[s["name"] for s in suspects],
        layout=widgets.Layout(width="260px"),
    )
    submit_button = widgets.Button(description="Submit Verdict")
    warning_output = widgets.Output()

    def _submit(_btn):
        submit_button.disabled = True
        with warning_output:
            clear_output()

        try:
            submit_verdict(killer_dropdown.value)
        except Exception as error:
            with warning_output:
                clear_output()
                display(widgets.HTML(
                    f'<div class="velvet-locked" style="margin-top:10px;">'
                    f'Something went wrong sealing your verdict ({error}). Please try again.</div>'
                ))
            submit_button.disabled = False
            return

        render_current_panel()  # switches straight to the reveal, since verdict is now set

    submit_button.on_click(_submit)

    display(widgets.HTML('<div class="velvet-mono" style="margin-top:4px;">SELECT THE MURDERER</div>'))
    display(killer_dropdown)
    display(widgets.HTML('<div style="margin-top:18px;"></div>'))
    display(submit_button)
    display(warning_output)


# FIX (Problem 7-9): the reveal now shows exactly one stored explanation --
# hidden_memory.final_solution, written once during case generation -- for
# both a correct and an incorrect guess. No narrated LLM verdict, no score,
# no rank; just whether the player was right, and the full true story.
def _render_verdict_reveal(result: dict):
    solution = hidden_memory.final_solution

    if result["killer_correct"]:
        headline = "CONGRATULATIONS, DETECTIVE."
        subline = "You correctly identified the murderer."
    else:
        headline = "CASE CLOSED."
        subline = (
            f"That was not the correct murderer. You accused "
            f"<strong>{result['submitted_killer']}</strong> — "
            f"here is what actually happened."
        )

    case_title = hidden_memory.case_json.get("case_title", "")
    case_title_html = (
        f'<div class="velvet-title" style="color:#b8892b; font-size:15px; margin-top:2px;">&ldquo;{case_title}&rdquo;</div>'
        if case_title else ""
    )

    evidence_supporting = solution.get("evidence_supporting") or []
    evidence_html = "".join(
        f'<div style="margin-top:6px; line-height:1.6;">&bull; {item}</div>'
        for item in evidence_supporting
    )

    display(widgets.HTML(f"""
        <div class="velvet-card velvet-fade-in">
          <div class="velvet-h2">{headline}</div>
          {case_title_html}
          <div style="line-height:1.6; margin-top:8px;">{subline}</div>
        </div>

        <div class="velvet-card velvet-fade-in" style="margin-top:16px;">
          <div class="velvet-mono" style="margin-bottom:6px;">FINAL SOLUTION</div>
          <div class="velvet-muted">Murderer: {solution.get('murderer', hidden_memory.killer)}</div>
          <div class="velvet-muted" style="margin-top:4px;">Motive: {solution.get('motive', '')}</div>
          <div style="line-height:1.6; margin-top:10px;">{solution.get('how_it_happened', '')}</div>
        </div>

        <div class="velvet-card velvet-fade-in" style="margin-top:16px;">
          <div class="velvet-mono" style="margin-bottom:6px;">EVIDENCE SUPPORTING THIS</div>
          {evidence_html}
        </div>

        <div class="velvet-muted velvet-fade-in" style="margin-top:14px;">Puzzle points: {investigation_memory.total_puzzle_points}</div>
    """))

    restart_button = widgets.Button(description="Start a New Investigation")
    restart_button.layout = widgets.Layout(margin="16px 0 0 0")
    restart_button.on_click(_restart_investigation)
    display(restart_button)


RENDERERS = {
    "crime_scene": render_crime_scene_panel,  # FIX (Problem 2): was missing, caused KeyError
    "victim": render_victim_panel,
    "suspects": render_suspects_panel,
    "evidence": render_evidence_panel,
    "log": render_log_panel,
    "verdict": render_verdict_panel,
}

print("✅ Verdict panel ready. Panel renderer registry complete.")


✅ Verdict panel ready. Panel renderer registry complete.


## 14 · Launch

The only cell you actually run to play. It assembles the fixed-size
`velvet-app` shell, shows the Home Screen, and wires the full flow:
**Begin Investigation → generate case (silent) → reveal crime scene
once → Continue → Investigation Hub.**

In [ ]:
app_shell = widgets.VBox(layout=widgets.Layout(width="960px", height="620px"))
app_shell.add_class("velvet-app")

crime_scene_output = widgets.Output()
continue_button = widgets.Button(description="Continue to Investigation Hub")
# FIX (Problem 12 cleanup): a standalone "🖼 View Crime Scene" button used
# to be created here, but it was never added to any container, so it was
# dead code — impossible to click. The Crime Scene tab in the hub navbar
# (Section 13c / 13c-ii) is the real, reachable way to revisit the image
# now, so this button and its handler have been removed instead of wiring
# up a second, redundant way to do the same thing.
continue_row = widgets.HBox([continue_button], layout=widgets.Layout(padding="16px 60px"))
crime_scene_container = widgets.VBox([crime_scene_output, continue_row])


def _show_screen(screen_name: str):
    nav_state.screen = screen_name
    target = _SCREEN_CONTAINERS[screen_name]

    # Phase 4 polish: each screen swap drops in a fresh element carrying
    # `.velvet-fade-in`, so the whole screen gently fades/slides into view
    # instead of popping in instantly.
    target.remove_class("velvet-fade-in")
    app_shell.children = [target]
    target.add_class("velvet-fade-in")

    if screen_name == "hub":
        switch_panel("victim")


def _on_begin(_button):
    """Generate a brand new case end-to-end.

    FIX (Problem 14): this used to have no error handling at all. If
    anything failed partway through — a flaky GitHub/OpenRouter/image
    request, or generate_case exhausting its retries — the exception
    propagated straight out of the button click handler, leaving
    begin_button permanently disabled with no explanation and no way to
    retry short of restarting the kernel. Now every failure is caught,
    reported in place, and the button is always re-enabled so the player
    can just try again.
    """
    global hidden_memory, investigation_memory, loaded_puzzles, crime_scene_path

    begin_button.disabled = True
    try:
        _set_home_status("Pulling case files from the archive...")
        all_suspects = load_characters("suspects")
        all_victims = load_characters("victims")
        loaded_puzzles = load_all_puzzles()
        # FIX (Problem 8): only offer categories that actually have
        # puzzles in them, so evidence is never assigned to an empty
        # category in the first place.
        puzzle_categories = usable_puzzle_categories(loaded_puzzles)

        _set_home_status("Casting the victim and three suspects...")
        victim, suspects, cast_info = select_cast(all_victims, all_suspects)

        _set_home_status("Writing the case file (this can take a moment)...")
        hidden_memory = generate_case(victim, suspects, puzzle_categories)
        investigation_memory = InvestigationMemory()

        _set_home_status("Rendering the crime scene...")
        with crime_scene_output:
            clear_output()
            crime_scene_path = generate_crime_scene(hidden_memory)

        _set_home_status("")
        # FIX (Problem 1): the flow used to jump straight to the standalone
        # crime-scene screen after case generation, which is the confusing
        # behavior this fixes. The crime scene image is still generated
        # here (unchanged), but the player now lands on the Investigation
        # Hub with the Victim panel active first (see _show_screen("hub")
        # below, which already defaults to the "victim" panel). Navigation
        # itself — the tab bar, the Crime Scene tab, etc. — is untouched.
        _show_screen("hub")
    except Exception as error:
        hidden_memory = None
        investigation_memory = None
        with crime_scene_output:
            clear_output()
        _set_home_status(
            f"Couldn't generate this case ({error}). Please try again.",
            tone="warning",
        )
    finally:
        begin_button.disabled = False


def _on_continue(_button):
    _show_screen("hub")


def _restart_investigation(_button=None):
    """Tear down the current case and return to the Home Screen so the
    player can begin an entirely new, freshly-generated investigation
    without restarting the notebook kernel.
    """
    global hidden_memory, investigation_memory

    hidden_memory = None
    investigation_memory = None
    nav_state.screen = "home"
    nav_state.panel = "victim"
    nav_state.open_suspect = None
    nav_state.open_evidence = None

    crime_scene_output.clear_output()
    begin_button.disabled = False
    _set_home_status("")
    _show_screen("home")


begin_button.on_click(_on_begin)
continue_button.on_click(_on_continue)

# --- Task 2: loading notice -----------------------------------------------
# FIX: running inside a notebook, the kernel's display() calls return
# immediately, but the actual ipywidgets comm round-trip that turns those
# calls into real, clickable DOM elements in the browser can lag a few
# seconds behind -- especially on the very first render of a session. A
# player (or judge/teacher) who starts interacting during that gap sees
# what looks like a broken/blank app. This notice sets expectations while
# that gap closes, then removes itself automatically once controls are
# actually ready -- it never needs to be dismissed by hand.
LOADING_NOTICE_TEXT = (
    "Please wait... Interactive controls are loading. Since this demo runs "
    "inside a Jupyter Notebook (.ipynb), buttons and input boxes may take a "
    "few seconds to appear after the page loads."
)
loading_notice = widgets.HTML(f'<div class="velvet-loading-notice">{LOADING_NOTICE_TEXT}</div>')

_loading_notice_hidden = False

def _hide_loading_notice(*_args):
    global _loading_notice_hidden
    if _loading_notice_hidden:
        return
    _loading_notice_hidden = True
    loading_notice.layout.display = "none"

# Best case: ipywidgets calls this the moment begin_button's view actually
# exists in the browser DOM -- i.e. the moment controls are genuinely
# ready -- so the notice disappears exactly when it should, not a fixed
# guess. on_displayed is a real DOMWidget hook, not a polling loop.
try:
    begin_button.on_displayed(_hide_loading_notice)
except Exception:
    pass

# Safety net: on_displayed isn't guaranteed to fire on every frontend/widget
# manager (Colab's custom comm layer in particular doesn't always trigger
# it), so this notice must never get stuck on screen permanently. Hide it
# unconditionally a few seconds after boot regardless of the callback above.
threading.Timer(4.0, _hide_loading_notice).start()

# --- Boot the app --------------------------------------------------------
print("⏳ Loading Interactive Components...This demo is running inside a Jupyter Notebook,\n"
"so buttons and input fields may take a few seconds to render.\n"
 "⚠️ IMPORTANT DEMO NOTE\n\nThis application runs inside a Jupyter Notebook (.ipynb) environment. Interactive JavaScript widgets may take a few seconds to load and render. If a button, text box, or dropdown does not appear immediately, please wait a moment—it is still loading.\n\n• Evidence: The 'Write Your Answer' text box may take a few seconds to appear. The crime description is located directly below the crime scene image and may also take a moment to load.\n\n• Suspects: The buttons for selecting the three suspects may appear after a short delay. Once a suspect is opened, the Investigation Notes are displayed at the bottom of the suspect page and may also take a few moments to render.\n\n• Verdict: The murderer selection dropdown and the Submit Verdict button may take a little longer to appear than the rest of the interface.\n\nIf any interactive component has not appeared yet, please wait a few seconds. In some cases, switching to another tab (such as Investigation or Evidence) and then returning to the current page will complete the rendering.\n\nThank you for your patience!")
_SCREEN_CONTAINERS = {"home": home_container, "crime_scene": crime_scene_container, "hub": hub_container}
display(loading_notice)
display(style_injector)
_show_screen("home")
display(app_shell)


⏳ Loading Interactive Components...This demo is running inside a Jupyter Notebook,
so buttons and input fields may take a few seconds to render.
If they do not appear immediately,
please wait a moment or briefly switch to another tab (such as Evidence or Investigation) and return to this page. The controls should appear automatically.⚠️ IMPORTANT DEMO NOTE

This application runs inside a Jupyter Notebook (.ipynb) environment. Interactive JavaScript widgets may take a few seconds to load and render. If a button, text box, or dropdown does not appear immediately, please wait a moment—it is still loading.

• Evidence: The 'Write Your Answer' text box may take a few seconds to appear. The crime description is located directly below the crime scene image and may also take a moment to load.

• Suspects: The buttons for selecting the three suspects may appear after a short delay. Once a suspect is opened, the Investigation Notes are displayed at the bottom of the suspect page and may also tak

HTML(value='<div class="velvet-loading-notice">Please wait... Interactive controls are loading. Since this dem…

HTML(value="\n<style>\n@import url('https://fonts.googleapis.com/css2?family=Special+Elite&family=Crimson+Pro:…